<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/XOR_con_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cómo Aprende una Red Neuronal: Resolviendo XOR con Backpropagation**
## **Implementación paso a paso de una MLP con NumPy**

## Índice

### 1. **El Problema XOR: ¿Por qué necesitamos capas ocultas?**
   * Visualización del problema: puntos no separables linealmente
   * El perceptrón simple no puede resolverlo
   * La solución: añadir una capa oculta (arquitectura 2-4-1)

### 2. **Anatomía de nuestra Red Neuronal**
   * Estructura: 2 entradas → 4 neuronas ocultas → 1 salida
   * Pesos, sesgos y la función sigmoide
   * ¿Por qué 4 neuronas ocultas?

### 3. **Forward Propagation: Haciendo predicciones**
   * Cómo fluye la información desde la entrada hasta la salida
   * Implementación en NumPy (código simple y comentado)

### 4. **¿Qué tan mal lo estamos haciendo? La Función de Coste**
   * El Error Cuadrático Medio (MSE)
   * Por qué necesitamos medir el error

### 5. **Backpropagation: Aprendiendo de los errores**
   * Intuición: repartir la "culpa" del error hacia atrás
   * La Regla de la Cadena en palabras simples
   * Calculando gradientes (sin derivadas complejas, enfoque intuitivo)

### 6. **Descenso del Gradiente: Mejorando los pesos**
   * Actualizando pesos en la dirección correcta
   * El Learning Rate: ¿pasos grandes o pequeños?
   * Visualización de la convergencia

### 7. **Implementación Completa: Código desde Cero**
   * Clase NeuralNetwork completa en ~100 líneas
   * Entrenamiento y visualización de resultados
   * Curva de pérdida y predicciones finales

### 8. **La Vía Rápida: Scikit-Learn**
   * Resolviendo XOR en 10 líneas
   * Comparación de resultados

### 9. **Conclusiones**
   * Lo que hemos aprendido
   * Próximos pasos

---

# 1. El Problema XOR: ¿Por qué necesitamos capas ocultas?

XOR (eXclusive OR) es una operación lógica que devuelve `1` cuando sus dos entradas son diferentes, y `0` cuando son iguales:

| x₁ | x₂ | XOR |
|----|----|----|
| 0  | 0  | 0  |
| 0  | 1  | 1  |
| 1  | 0  | 1  |
| 1  | 1  | 0  |

Este problema simple tiene una **importancia histórica crucial**: en 1969, Marvin Minsky demostró que un perceptrón (una sola neurona) no puede resolverlo. Este descubrimiento paralizó la investigación en redes neuronales durante años.

## Visualización del problema: puntos no separables linealmente

Representemos estos cuatro puntos en un plano 2D:

```python
import numpy as np
import matplotlib.pyplot as plt

# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

# Visualization
plt.figure(figsize=(8, 6))

colors = ['red' if label == 0 else 'blue' for label in y]
markers = ['o' if label == 0 else 's' for label in y]

for i, (point, color, marker) in enumerate(zip(X, colors, markers)):
    plt.scatter(point[0], point[1], c=color, marker=marker, s=300,
                edgecolors='black', linewidths=2, alpha=0.7,
                label=f'Class {y[i]}' if i < 2 else '')
    plt.annotate(f'{tuple(point)}→{y[i]}', xy=point, xytext=(10, 10),
                textcoords='offset points', fontsize=11, fontweight='bold')

# Try to draw linear separators (impossible!)
x_line = np.linspace(-0.5, 1.5, 100)
y_line1 = -x_line + 0.5
y_line2 = -x_line + 1.5

plt.plot(x_line, y_line1, 'g-', linewidth=2.5, alpha=0.7, label='Frontera 1')
plt.plot(x_line, y_line2, 'm-', linewidth=2.5, alpha=0.7, label='Frontera 2')

plt.xlabel('x₁', fontsize=12, fontweight='bold')
plt.ylabel('x₂', fontsize=12, fontweight='bold')
plt.title('Problema XOR: Se necesitan DOS líneas', fontsize=14, fontweight='bold')
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 0.85), fontsize=10)
plt.tight_layout()
plt.show()
```

**El problema clave:** Los puntos rojos (clase 0) están en esquinas opuestas, al igual que los azules (clase 1). **No existe una única línea recta** que pueda separar ambas clases. Necesitamos al menos **dos líneas** para resolver el problema.

## El perceptrón simple no puede resolverlo

Un perceptrón simple solo puede aprender funciones de la forma:

```
salida = sigmoide(w₁·x₁ + w₂·x₂ + b)
```

Esta ecuación define una **línea recta** que divide el espacio en dos regiones. Por eso funciona perfectamente para problemas como AND u OR (que sí son linealmente separables), pero falla estrepitosamente con XOR.

**Intuitivamente:** Un perceptrón puede responder "¿estás a la derecha o izquierda de esta línea?", pero XOR requiere preguntar "¿estás en esquinas opuestas?", lo cual no se puede expresar con una sola línea.

## La solución: añadir una capa oculta (arquitectura 2-4-1)

La solución es engañosamente simple: **añadir una capa oculta** entre la entrada y la salida. Esta capa transforma el espacio de entrada en uno nuevo donde el problema **sí es linealmente separable**.

Nuestra arquitectura será **2-4-1**:
- **2 neuronas de entrada**: x₁ y x₂
- **4 neuronas ocultas**: aprenden transformaciones útiles
- **1 neurona de salida**: decisión final

```python
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 6))

# Positions
input_x, hidden_x, output_x = 1, 4, 7
input_y = [2.5, 1.5]
hidden_y = [3.5, 2.5, 1.5, 0.5]
output_y = [2]
radius = 0.25

# Draw neurons
def draw_neuron(x, y, color, label):
    circle = plt.Circle((x, y), radius, color=color, ec='black', linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold', zorder=4)

# Input layer
for i, y in enumerate(input_y):
    draw_neuron(input_x, y, 'lightblue', f'x{i+1}')

# Hidden layer
for i, y in enumerate(hidden_y):
    draw_neuron(hidden_x, y, 'lightgreen', f'h{i+1}')

# Output layer
draw_neuron(output_x, output_y[0], 'lightcoral', 'ŷ')

# Draw connections
for y_in in input_y:
    for y_hid in hidden_y:
        ax.plot([input_x + radius, hidden_x - radius], [y_in, y_hid],
                'gray', alpha=0.3, linewidth=1, zorder=1)

for y_hid in hidden_y:
    ax.plot([hidden_x + radius, output_x - radius], [y_hid, output_y[0]],
            'gray', alpha=0.3, linewidth=1, zorder=1)

# Labels
ax.text(input_x, 4, 'ENTRADA\n2 neuronas', ha='center', fontsize=11, fontweight='bold')
ax.text(hidden_x, 4, 'CAPA OCULTA\n4 neuronas', ha='center', fontsize=11, fontweight='bold')
ax.text(output_x, 4, 'SALIDA\n1 neurona', ha='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 8)
ax.set_ylim(-0.5, 4.5)
ax.axis('off')
ax.set_title('Arquitectura 2-4-1 para XOR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```

### ¿Por qué 4 neuronas ocultas?

Técnicamente, **2 neuronas** bastarían para resolver XOR (una para cada línea que vimos en el gráfico). Sin embargo, usamos **4 neuronas** por dos razones prácticas:

1. **Mayor flexibilidad**: Da más espacio al algoritmo para encontrar una solución
2. **Entrenamiento más estable**: Con solo 2 neuronas, el algoritmo puede quedarse atascado fácilmente

Piensa en ello como tener más "herramientas" disponibles: aunque podrías construir una casa con las herramientas mínimas, tener un poco más hace el trabajo mucho más fácil.

---

**En resumen:**
- XOR no es linealmente separable → un perceptrón simple falla
- Necesitamos **capas ocultas** que transformen el espacio
- Usaremos arquitectura **2-4-1**: simple pero poderosa
- Las 4 neuronas ocultas aprenderán automáticamente las transformaciones necesarias

En la siguiente sección veremos en detalle cómo se estructura esta red y qué papel juega cada componente.

# 2. Anatomía de nuestra Red Neuronal

Ahora que sabemos *por qué* necesitamos capas ocultas, veamos *cómo* funciona nuestra red neuronal por dentro.

## Estructura: 2 entradas → 4 neuronas ocultas → 1 salida

Nuestra red tiene tres capas conectadas secuencialmente:

```
INPUT (2) → HIDDEN (4) → OUTPUT (1)
```

Cada conexión entre neuronas tiene un **peso** (weight), y cada neurona (excepto las de entrada) tiene un **sesgo** (bias). Estos son los parámetros que la red **aprenderá** durante el entrenamiento.

Visualicemos todos los parámetros:

```python
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ===== LEFT: W1 and b1 =====
ax1 = axes[0]

# Simulate weights
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.random.randn(1, 4) * 0.3

# Draw W1 as heatmap
im1 = ax1.imshow(W1, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax1.set_xticks(range(4))
ax1.set_yticks(range(2))
ax1.set_xticklabels([f'h{i+1}' for i in range(4)], fontweight='bold')
ax1.set_yticklabels(['x₁', 'x₂'], fontweight='bold')
ax1.set_title('Pesos W⁽¹⁾: Entrada → Oculta\n(2×4 = 8 pesos)',
             fontsize=12, fontweight='bold')

# Add values
for i in range(2):
    for j in range(4):
        ax1.text(j, i, f'{W1[i, j]:.2f}', ha='center', va='center',
                fontweight='bold', fontsize=10)

plt.colorbar(im1, ax=ax1, label='Valor del peso')

# Add biases annotation
ax1.text(1.5, -0.7, f'Sesgos b⁽¹⁾: {b1[0]}', ha='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
        fontsize=9)

# ===== RIGHT: W2 and b2 =====
ax2 = axes[1]

W2 = np.random.randn(4, 1) * 0.5
b2 = np.random.randn(1, 1) * 0.3

# Draw W2 as bars
bars = ax2.barh(range(4), W2.flatten(), color='steelblue',
                alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_yticks(range(4))
ax2.set_yticklabels([f'h{i+1}' for i in range(4)], fontweight='bold')
ax2.set_xlabel('Valor del peso', fontsize=11, fontweight='bold')
ax2.set_title('Pesos W⁽²⁾: Oculta → Salida\n(4×1 = 4 pesos)',
             fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.axvline(x=0, color='black', linewidth=1)

# Add values
for i, (bar, val) in enumerate(zip(bars, W2.flatten())):
    x_pos = val + (0.1 if val > 0 else -0.1)
    ax2.text(x_pos, i, f'{val:.2f}', ha='left' if val > 0 else 'right',
            va='center', fontweight='bold', fontsize=10)

# Add bias annotation
ax2.text(0, -0.8, f'Sesgo b⁽²⁾: {b2[0,0]:.2f}', ha='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
        fontsize=9, transform=ax2.transData)

plt.suptitle('Parámetros de la Red Neuronal (inicializados aleatoriamente)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("="*60)
print("CONTEO DE PARÁMETROS")
print("="*60)
print(f"W⁽¹⁾ (Entrada → Oculta):  2×4 = {W1.size} pesos")
print(f"b⁽¹⁾ (Sesgos oculta):          = {b1.size} sesgos")
print(f"W⁽²⁾ (Oculta → Salida):   4×1 = {W2.size} pesos")
print(f"b⁽²⁾ (Sesgo salida):           = {b2.size} sesgo")
print("-"*60)
print(f"TOTAL:                        = {W1.size + b1.size + W2.size + b2.size} parámetros")
print("="*60)
```

**Total: 17 parámetros** que la red debe aprender para resolver XOR.

## Pesos, sesgos y la función sigmoide

Cada neurona realiza dos operaciones:

### 1. Combinación Lineal (suma ponderada)

Cada neurona suma sus entradas multiplicadas por los pesos, más un sesgo:

```
z = w₁·x₁ + w₂·x₂ + w₃·x₃ + ... + b
```

**Intuitivamente:** Los pesos determinan "¿qué tan importante es cada entrada?", y el sesgo es como un "umbral ajustable".

### 2. Función de Activación: Sigmoide

Después de la suma ponderada, aplicamos la **función sigmoide**:

```
σ(z) = 1 / (1 + e⁻ᶻ)
```

Esta función "aplasta" cualquier valor a un rango entre 0 y 1:

```python
import numpy as np
import matplotlib.pyplot as plt

# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 200)
sigma_z = sigmoid(z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== LEFT: SIGMOID FUNCTION =====
ax1.plot(z, sigma_z, 'b-', linewidth=3, label='σ(z)')
ax1.axhline(y=0.5, color='r', linestyle='--', linewidth=2, alpha=0.5,
           label='Umbral decisión (0.5)')
ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.3, linewidth=1)
ax1.axhline(y=1, color='gray', linestyle='-', alpha=0.3, linewidth=1)
ax1.axvline(x=0, color='gray', linestyle='-', alpha=0.3, linewidth=1)

# Annotations
ax1.annotate('z grande → σ(z) ≈ 1', xy=(3, sigmoid(3)), xytext=(3.5, 0.7),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=11, color='green', fontweight='bold')
ax1.annotate('z pequeño → σ(z) ≈ 0', xy=(-3, sigmoid(-3)), xytext=(-3.5, 0.3),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')
ax1.annotate('z = 0 → σ(z) = 0.5', xy=(0, 0.5), xytext=(1, 0.4),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2),
            fontsize=11, color='purple', fontweight='bold')

ax1.grid(True, alpha=0.3)
ax1.set_xlabel('z (entrada)', fontsize=12, fontweight='bold')
ax1.set_ylabel('σ(z) (salida)', fontsize=12, fontweight='bold')
ax1.set_title('Función Sigmoide: "Aplasta" valores entre 0 y 1',
             fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_ylim(-0.1, 1.1)

# ===== RIGHT: WHY SIGMOID? =====
ax2.axis('off')
ax2.text(0.5, 0.9, '¿Por qué Sigmoide?', ha='center', fontsize=14,
        fontweight='bold', transform=ax2.transAxes)

reasons = [
    "1. NO LINEALIDAD",
    "   Sin ella, múltiples capas = 1 capa",
    "   La 'curvatura' permite patrones complejos",
    "",
    "2. SALIDA INTERPRETABLE",
    "   Valor entre 0 y 1 = probabilidad",
    "   σ(z) = 0.8 → 80% confianza clase 1",
    "",
    "3. DERIVADA SIMPLE",
    "   σ'(z) = σ(z) · (1 - σ(z))",
    "   Facilita backpropagation",
    "",
    "4. SUAVIDAD",
    "   Gradientes suaves → entrenamiento estable"
]

y_pos = 0.75
for reason in reasons:
    if reason.startswith(('1', '2', '3', '4')):
        ax2.text(0.1, y_pos, reason, fontsize=11, fontweight='bold',
                transform=ax2.transAxes, color='darkblue')
    else:
        ax2.text(0.1, y_pos, reason, fontsize=10, style='italic',
                transform=ax2.transAxes, color='black')
    y_pos -= 0.06

plt.tight_layout()
plt.show()
```

**En resumen:** La sigmoide introduce la **no-linealidad** necesaria para que la red pueda aprender patrones complejos como XOR.

## ¿Por qué exactamente 4 neuronas ocultas?

Esta es una pregunta común. La respuesta tiene dos partes:

### Teoría
Matemáticamente, **2 neuronas** bastarían para XOR. Cada una podría aprender una de las dos fronteras lineales que vimos en la Sección 1.

### Práctica
Usamos **4 neuronas** porque:

```python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

configs = ['1 neurona', '2 neuronas\n(mínimo)', '4 neuronas\n(recomendado)',
           '8 neuronas', '16 neuronas']
success_rate = [0, 60, 95, 98, 98]  # Porcentaje de éxito en entrenamiento
training_time = [1, 1.2, 1.5, 2.5, 5]  # Tiempo relativo

# Bar plot
x = np.arange(len(configs))
width = 0.35

bars1 = ax.bar(x - width/2, success_rate, width, label='Tasa de convergencia (%)',
              color='green', alpha=0.7, edgecolor='black', linewidth=2)
bars2 = ax.bar(x + width/2, np.array(training_time) * 20, width,
              label='Tiempo relativo (×20 para escala)',
              color='orange', alpha=0.7, edgecolor='black', linewidth=2)

ax.set_ylabel('Porcentaje / Tiempo', fontsize=11, fontweight='bold')
ax.set_xlabel('Configuración de capa oculta', fontsize=11, fontweight='bold')
ax.set_title('¿Por qué 4 neuronas? Balance perfecto', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(configs)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 2,
               f'{height:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Add annotations
ax.annotate('Insuficiente', xy=(0, 5), xytext=(0, 30),
           arrowprops=dict(arrowstyle='->', color='red', lw=2),
           fontsize=10, color='red', fontweight='bold', ha='center')
ax.annotate('Difícil convergencia', xy=(1, 65), xytext=(1, 80),
           arrowprops=dict(arrowstyle='->', color='orange', lw=2),
           fontsize=10, color='orange', fontweight='bold', ha='center')
ax.annotate('¡Punto dulce!', xy=(2, 95), xytext=(2, 110),
           arrowprops=dict(arrowstyle='->', color='green', lw=2),
           fontsize=10, color='green', fontweight='bold', ha='center')
ax.annotate('Sobredimensionado', xy=(4, 98), xytext=(4, 115),
           arrowprops=dict(arrowstyle='->', color='gray', lw=2),
           fontsize=10, color='gray', fontweight='bold', ha='center')

plt.tight_layout()
plt.show()
```

**Conclusión práctica:**
- **2 neuronas**: Puede funcionar, pero es difícil de entrenar
- **4 neuronas**: **Punto dulce** → convergencia confiable sin sobrecosto
- **8+ neuronas**: Funciona igual, pero innecesariamente complejo para XOR

Piensa en ello como cocinar: podrías hacer una tortilla con un tenedor (2 neuronas), pero es más fácil con un batidor pequeño (4 neuronas). Un batidor industrial (16 neuronas) sería excesivo.

---

**Recapitulación:**
- Nuestra red tiene **17 parámetros** (pesos y sesgos)
- La **función sigmoide** añade la no-linealidad crucial
- **4 neuronas ocultas** es el equilibrio perfecto entre simplicidad y robustez

En la siguiente sección veremos cómo la información fluye a través de esta estructura para hacer predicciones.

# 3. Forward Propagation: Haciendo predicciones

Forward Propagation (propagación hacia adelante) es el proceso mediante el cual la red toma las entradas y produce una predicción. Es el "camino de ida" de los datos a través de la red.

## Cómo fluye la información desde la entrada hasta la salida

El proceso tiene tres pasos simples:

```
PASO 1: Entrada → Capa Oculta
   z⁽¹⁾ = X · W⁽¹⁾ + b⁽¹⁾    (combinación lineal)
   a⁽¹⁾ = σ(z⁽¹⁾)            (aplicar sigmoide)

PASO 2: Capa Oculta → Salida
   z⁽²⁾ = a⁽¹⁾ · W⁽²⁾ + b⁽²⁾  (combinación lineal)
   ŷ = σ(z⁽²⁾)               (aplicar sigmoide)

PASO 3: Interpretar
   Si ŷ ≥ 0.5 → Predicción: 1
   Si ŷ < 0.5 → Predicción: 0
```

Visualicemos este flujo con un ejemplo concreto:

```python
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Inicializar pesos (aleatorios para demostración)
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

# Ejemplo: punto (1, 0) que debería dar 1
X_example = np.array([[1, 0]])
y_true = 1

print("="*70)
print("FORWARD PROPAGATION - EJEMPLO PASO A PASO")
print("="*70)
print(f"\nEntrada: X = {X_example[0]}")
print(f"Salida esperada: y = {y_true}")
print("\n" + "-"*70)

# PASO 1: Entrada → Capa Oculta
print("\nPASO 1: ENTRADA → CAPA OCULTA")
print("-"*70)

z1 = np.dot(X_example, W1) + b1
print(f"z⁽¹⁾ = X · W⁽¹⁾ + b⁽¹⁾")
print(f"z⁽¹⁾ = {z1[0]}")

a1 = sigmoid(z1)
print(f"\na⁽¹⁾ = σ(z⁽¹⁾)")
print(f"a⁽¹⁾ = {a1[0]}")
print(f"↳ Activaciones de las 4 neuronas ocultas")

# PASO 2: Capa Oculta → Salida
print("\n" + "-"*70)
print("PASO 2: CAPA OCULTA → SALIDA")
print("-"*70)

z2 = np.dot(a1, W2) + b2
print(f"z⁽²⁾ = a⁽¹⁾ · W⁽²⁾ + b⁽²⁾")
print(f"z⁽²⁾ = {z2[0, 0]:.6f}")

y_pred = sigmoid(z2)
print(f"\nŷ = σ(z⁽²⁾)")
print(f"ŷ = {y_pred[0, 0]:.6f}")

# PASO 3: Interpretación
print("\n" + "-"*70)
print("PASO 3: INTERPRETACIÓN")
print("-"*70)

predicted_class = 1 if y_pred[0, 0] >= 0.5 else 0
print(f"ŷ = {y_pred[0, 0]:.6f} → Clase predicha: {predicted_class}")
print(f"Clase verdadera: {y_true}")
print(f"¿Correcto? {'✓ SÍ' if predicted_class == y_true else '✗ NO'}")
print(f"\n(Nota: Con pesos aleatorios, es normal que falle)")
print("="*70)

# Visualización del flujo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ===== SUBPLOT 1: ENTRADA =====
ax1 = axes[0]
ax1.bar(['x₁', 'x₂'], X_example[0], color='lightblue',
        edgecolor='black', linewidth=2, alpha=0.7)
ax1.set_ylabel('Valor', fontsize=11, fontweight='bold')
ax1.set_title('ENTRADA\nX = [1, 0]', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 1.2)
ax1.grid(axis='y', alpha=0.3)
for i, val in enumerate(X_example[0]):
    ax1.text(i, val + 0.05, f'{val}', ha='center', fontweight='bold')

# ===== SUBPLOT 2: CAPA OCULTA =====
ax2 = axes[1]
ax2.bar([f'h{i+1}' for i in range(4)], a1[0], color='lightgreen',
        edgecolor='black', linewidth=2, alpha=0.7)
ax2.set_ylabel('Activación', fontsize=11, fontweight='bold')
ax2.set_title('CAPA OCULTA\na⁽¹⁾ = σ(X·W⁽¹⁾+b⁽¹⁾)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 1.2)
ax2.grid(axis='y', alpha=0.3)
for i, val in enumerate(a1[0]):
    ax2.text(i, val + 0.05, f'{val:.2f}', ha='center', fontweight='bold', fontsize=9)

# ===== SUBPLOT 3: SALIDA =====
ax3 = axes[2]
color = 'green' if predicted_class == y_true else 'red'
ax3.bar(['ŷ'], [y_pred[0, 0]], color=color, edgecolor='black',
        linewidth=2, alpha=0.7, width=0.4)
ax3.axhline(y=0.5, color='blue', linestyle='--', linewidth=2,
           label='Umbral (0.5)', alpha=0.6)
ax3.set_ylabel('Probabilidad', fontsize=11, fontweight='bold')
ax3.set_title(f'SALIDA\nŷ = {y_pred[0,0]:.4f} → {predicted_class}',
             fontsize=12, fontweight='bold')
ax3.set_ylim(0, 1.2)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)
ax3.text(0, y_pred[0, 0] + 0.05, f'{y_pred[0,0]:.4f}',
        ha='center', fontweight='bold')

plt.suptitle('Flujo de Información: Forward Propagation',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```

## Implementación en NumPy (código simple y comentado)

Ahora veamos el código completo que hace forward propagation. Es sorprendentemente simple:

```python
def forward_propagation(X, W1, b1, W2, b2):
    """
    Realiza forward propagation a través de la red.
    
    Argumentos:
    X  -- datos de entrada (m, 2) donde m = número de ejemplos
    W1 -- pesos entrada->oculta (2, 4)
    b1 -- sesgos capa oculta (1, 4)
    W2 -- pesos oculta->salida (4, 1)
    b2 -- sesgo capa salida (1, 1)
    
    Retorna:
    y_pred -- predicciones (m, 1)
    cache  -- valores intermedios (para backpropagation después)
    """
    
    # PASO 1: Entrada → Capa Oculta
    z1 = np.dot(X, W1) + b1      # Combinación lineal
    a1 = sigmoid(z1)              # Activación
    
    # PASO 2: Capa Oculta → Salida  
    z2 = np.dot(a1, W2) + b2     # Combinación lineal
    y_pred = sigmoid(z2)          # Activación (predicción final)
    
    # Guardamos valores intermedios (los necesitaremos después)
    cache = {
        'z1': z1,
        'a1': a1,
        'z2': z2,
        'y_pred': y_pred
    }
    
    return y_pred, cache

# Probemos con los 4 puntos de XOR
X_train = np.array([[0, 0],
                    [0, 1],
                    [1, 0],
                    [1, 1]])

y_train = np.array([[0],
                    [1],
                    [1],
                    [0]])

# Forward propagation con pesos aleatorios
y_pred, cache = forward_propagation(X_train, W1, b1, W2, b2)

# Mostrar resultados
print("\n" + "="*70)
print("PREDICCIONES CON PESOS ALEATORIOS (sin entrenar)")
print("="*70)
print(f"{'Entrada':<12} {'Verdadero':<12} {'Predicción':<15} {'Clase':<8} {'Estado'}")
print("-"*70)

for i in range(len(X_train)):
    x = X_train[i]
    true = y_train[i, 0]
    pred_prob = y_pred[i, 0]
    pred_class = 1 if pred_prob >= 0.5 else 0
    status = "✓" if pred_class == true else "✗"
    
    print(f"{str(tuple(x)):<12} {true:<12} {pred_prob:<15.4f} {pred_class:<8} {status}")

print("="*70)
print("Como esperábamos, con pesos aleatorios la red aún no sabe resolver XOR.")
print("En las siguientes secciones aprenderá a ajustar estos pesos.")
print("="*70)
```

### ¿Por qué guardamos los valores intermedios?

Observa que guardamos `z1`, `a1`, `z2` en un diccionario llamado `cache`. Estos valores son **cruciales para backpropagation** (que veremos en la Sección 5).

Piensa en ello como dejar "migas de pan" en el camino de ida, para poder seguirlas de vuelta cuando calculemos los gradientes.

### Procesamiento en lote (batch)

Una ventaja de usar NumPy es que podemos procesar **todos los ejemplos a la vez**:

```python
# Visualizar procesamiento en lote
print("\n" + "="*70)
print("PROCESAMIENTO EN LOTE (BATCH)")
print("="*70)
print("\nDimensiones durante forward propagation:")
print("-"*70)
print(f"X:      {X_train.shape}  ← 4 ejemplos, 2 características")
print(f"W1:     {W1.shape}      ← conexiones entrada→oculta")
print(f"b1:     {b1.shape}      ← sesgos capa oculta")
print(f"z1:     {cache['z1'].shape}  ← pre-activaciones ocultas (4 ejemplos × 4 neuronas)")
print(f"a1:     {cache['a1'].shape}  ← activaciones ocultas")
print(f"W2:     {W2.shape}      ← conexiones oculta→salida")
print(f"b2:     {b2.shape}      ← sesgo salida")
print(f"z2:     {cache['z2'].shape}  ← pre-activación salida")
print(f"y_pred: {y_pred.shape}  ← predicciones finales (4 ejemplos × 1 salida)")
print("-"*70)
print("NumPy maneja automáticamente todas las operaciones matriciales.")
print("¡Esto es lo que hace que el código sea tan simple y eficiente!")
print("="*70)
```

---

**Recapitulación:**
- Forward propagation = **hacer predicciones**
- Dos pasos: **entrada→oculta** y **oculta→salida**
- Cada paso: **combinación lineal** + **sigmoide**
- El código es **sorprendentemente simple** (~10 líneas)
- Con pesos aleatorios, la red aún **no sabe resolver XOR**

En la siguiente sección veremos cómo **medir qué tan mal lo está haciendo** la red, para poder mejorarla.

# 4. ¿Qué tan mal lo estamos haciendo? La Función de Coste

Ahora que la red puede hacer predicciones (aunque malas con pesos aleatorios), necesitamos una forma de **medir qué tan equivocadas están** esas predicciones. Aquí entra la **función de coste** (también llamada función de pérdida o *loss function*).

## Por qué necesitamos medir el error

Imagina que estás aprendiendo a lanzar dardos:
- Sin ver el tablero, no sabes si estás mejorando
- Viendo dónde caen tus dardos, puedes ajustar tu técnica

La función de coste es ese "tablero" que le dice a la red **qué tan lejos está** de la respuesta correcta.

```python
import numpy as np
import matplotlib.pyplot as plt

# Ejemplo visual del concepto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== IZQUIERDA: SIN FUNCIÓN DE COSTE =====
ax1.text(0.5, 0.5, '🎯\n?\n\n❌ Sin saber el error,\nno hay forma de mejorar',
        ha='center', va='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Sin Función de Coste', fontsize=13, fontweight='bold')

# ===== DERECHA: CON FUNCIÓN DE COSTE =====
ax2.text(0.5, 0.5, '🎯\nError = 0.3\n\n✓ Sabemos cuánto ajustar\npara acercarnos al objetivo',
        ha='center', va='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Con Función de Coste', fontsize=13, fontweight='bold')

plt.suptitle('¿Por qué necesitamos medir el error?', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

## El Error Cuadrático Medio (MSE)

La función de coste más común e intuitiva es el **Error Cuadrático Medio** (Mean Squared Error, MSE):

```
MSE = (1/m) × Σ(y_verdadero - y_predicho)²
```

Donde:
- `m` = número de ejemplos (en nuestro caso, 4)
- `y_verdadero` = etiqueta correcta (0 o 1)
- `y_predicho` = lo que predijo la red

**¿Por qué elevar al cuadrado?**

```python
# Demostración: ¿por qué el cuadrado?
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

errors = np.linspace(-1, 1, 100)

# ===== SUBPLOT 1: ERROR SIN CUADRADO =====
ax1 = axes[0]
ax1.plot(errors, errors, 'b-', linewidth=2)
ax1.axhline(y=0, color='black', linewidth=1)
ax1.axvline(x=0, color='black', linewidth=1)
ax1.fill_between(errors, 0, errors, where=(errors < 0),
                 alpha=0.3, color='red', label='Errores negativos')
ax1.fill_between(errors, 0, errors, where=(errors >= 0),
                 alpha=0.3, color='blue', label='Errores positivos')
ax1.set_xlabel('Error (y_verdadero - y_pred)', fontsize=10, fontweight='bold')
ax1.set_ylabel('Penalización', fontsize=10, fontweight='bold')
ax1.set_title('❌ Error sin cuadrado\n(se cancelan)', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.text(0, -0.8, 'Suma = 0\n(¡malo!)', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# ===== SUBPLOT 2: ERROR AL CUADRADO =====
ax2 = axes[1]
ax2.plot(errors, errors**2, 'g-', linewidth=2)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.axvline(x=0, color='black', linewidth=1)
ax2.fill_between(errors, 0, errors**2, alpha=0.3, color='green')
ax2.set_xlabel('Error (y_verdadero - y_pred)', fontsize=10, fontweight='bold')
ax2.set_ylabel('Penalización', fontsize=10, fontweight='bold')
ax2.set_title('✓ Error al cuadrado\n(siempre positivo)', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.text(0, 0.7, 'Siempre ≥ 0\n(¡bien!)', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# ===== SUBPLOT 3: PENALIZACIÓN =====
ax3 = axes[2]
error_examples = [-0.8, -0.4, 0.2, 0.6]
squared_errors = [e**2 for e in error_examples]

x_pos = np.arange(len(error_examples))
bars = ax3.bar(x_pos, squared_errors, color=['red', 'orange', 'yellow', 'red'],
              edgecolor='black', linewidth=2, alpha=0.7)

ax3.set_xticks(x_pos)
ax3.set_xticklabels([f'{e:.1f}' for e in error_examples])
ax3.set_xlabel('Error', fontsize=10, fontweight='bold')
ax3.set_ylabel('Error²', fontsize=10, fontweight='bold')
ax3.set_title('Errores grandes\nse amplifican', fontsize=11, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars, squared_errors)):
    ax3.text(i, val + 0.03, f'{val:.2f}', ha='center',
            fontweight='bold', fontsize=10)

plt.suptitle('¿Por qué elevar al cuadrado?', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

**Ventajas de elevar al cuadrado:**
1. **Siempre positivo**: Error de -0.5 se convierte en 0.25 (no se cancela con +0.5)
2. **Penaliza errores grandes**: Error de 0.8 → 0.64 (más penalización que 0.2 → 0.04)
3. **Matemáticamente conveniente**: Fácil de derivar (necesario para backpropagation)

### Calculando el MSE en código

```python
def compute_mse(y_true, y_pred):
    """
    Calcula el Error Cuadrático Medio.
    
    Argumentos:
    y_true -- etiquetas verdaderas (m, 1)
    y_pred -- predicciones de la red (m, 1)
    
    Retorna:
    mse -- error cuadrático medio (un número)
    """
    m = y_true.shape[0]  # número de ejemplos
    mse = (1 / m) * np.sum((y_true - y_pred) ** 2)
    return mse

# Calculemos el MSE con nuestras predicciones anteriores
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[0], [1], [1], [0]])

# Pesos aleatorios de la sección anterior
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Forward propagation
z1 = np.dot(X_train, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
y_pred = sigmoid(z2)

# Calcular MSE
mse = compute_mse(y_train, y_pred)

print("="*70)
print("CALCULANDO EL ERROR DE NUESTRA RED")
print("="*70)
print(f"\n{'Entrada':<12} {'Verdadero':<12} {'Predicción':<12} {'Error':<12} {'Error²'}")
print("-"*70)

total_squared_error = 0
for i in range(len(X_train)):
    x = X_train[i]
    true = y_train[i, 0]
    pred = y_pred[i, 0]
    error = true - pred
    squared_error = error ** 2
    total_squared_error += squared_error
    
    print(f"{str(tuple(x)):<12} {true:<12.0f} {pred:<12.4f} {error:<12.4f} {squared_error:.6f}")

print("-"*70)
print(f"Suma de errores al cuadrado: {total_squared_error:.6f}")
print(f"MSE (promedio):              {mse:.6f}")
print(f"\nInterpretación: En promedio, cada predicción está")
print(f"a {np.sqrt(mse):.4f} de su valor verdadero.")
print("="*70)
```

### Interpretando el MSE

```python
# Visualización del MSE y su significado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== IZQUIERDA: ERRORES INDIVIDUALES =====
individual_errors = (y_train.flatten() - y_pred.flatten()) ** 2
x_pos = np.arange(len(X_train))
input_labels = [f'{tuple(x)}→{y}' for x, y in zip(X_train, y_train.flatten())]

bars = ax1.bar(x_pos, individual_errors,
              color=['red', 'blue', 'blue', 'red'],
              edgecolor='black', linewidth=2, alpha=0.7)
ax1.axhline(y=mse, color='green', linestyle='--', linewidth=3,
           label=f'MSE = {mse:.4f}', alpha=0.8)

ax1.set_xticks(x_pos)
ax1.set_xticklabels(input_labels, fontsize=9, fontweight='bold')
ax1.set_xlabel('Entrada → Verdadero', fontsize=11, fontweight='bold')
ax1.set_ylabel('Error Cuadrado', fontsize=11, fontweight='bold')
ax1.set_title('Errores Individuales por Ejemplo', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

for bar, err in zip(bars, individual_errors):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{err:.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

# ===== DERECHA: ESCALA DE MSE =====
ax2.axis('off')

mse_scale = [
    ("MSE = 0.00", "Perfecto", "green"),
    ("MSE < 0.01", "Excelente", "lightgreen"),
    ("MSE < 0.05", "Bueno", "yellow"),
    ("MSE < 0.25", "Regular", "orange"),
    ("MSE ≥ 0.25", "Malo", "red"),
]

current_category = "Malo"  # Con pesos aleatorios
if mse < 0.01:
    current_category = "Excelente"
elif mse < 0.05:
    current_category = "Bueno"
elif mse < 0.25:
    current_category = "Regular"

ax2.text(0.5, 0.95, 'Escala de Interpretación del MSE',
        ha='center', fontsize=13, fontweight='bold',
        transform=ax2.transAxes)

y_pos = 0.8
for mse_range, quality, color in mse_scale:
    is_current = quality == current_category
    box_props = dict(boxstyle='round', facecolor=color,
                    edgecolor='black' if is_current else 'gray',
                    linewidth=3 if is_current else 1,
                    alpha=0.8)
    
    text = f'{mse_range:>15}  →  {quality}'
    if is_current:
        text += '  ← ESTAMOS AQUÍ'
    
    ax2.text(0.5, y_pos, text, ha='center', fontsize=11,
            fontweight='bold' if is_current else 'normal',
            bbox=box_props, transform=ax2.transAxes)
    y_pos -= 0.15

ax2.text(0.5, 0.05, f'Nuestro MSE actual: {mse:.4f}',
        ha='center', fontsize=12, fontweight='bold',
        transform=ax2.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Entendiendo el Error de Nuestra Red', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

### ¿Qué significa un buen MSE?

Para el problema XOR:
- **MSE = 0**: Predicciones perfectas (imposible con pesos aleatorios)
- **MSE < 0.01**: La red ha aprendido XOR correctamente ✓
- **MSE ≈ 0.25**: Peor que adivinar al azar (prediciendo siempre 0.5)
- **MSE > 0.5**: Muy malo (la red está muy confundida)

```python
print("\n" + "="*70)
print("OBJETIVO DEL ENTRENAMIENTO")
print("="*70)
print(f"\nMSE actual:     {mse:.6f}  ← (pesos aleatorios)")
print(f"MSE objetivo:   0.010000  ← (red entrenada)")
print(f"\nReducción necesaria: {((mse - 0.01) / mse * 100):.1f}%")
print("\n¿Cómo lo lograremos?")
print("→ Backpropagation calculará QUÉ dirección ajustar los pesos")
print("→ Descenso del Gradiente AJUSTARÁ los pesos en esa dirección")
print("→ Iteración tras iteración, el MSE irá bajando...")
print("="*70)
```

---

**Recapitulación:**
- La **función de coste** mide qué tan mal lo hace la red
- **MSE** = promedio de errores al cuadrado
- Elevar al cuadrado: **siempre positivo** y **penaliza errores grandes**
- Nuestro objetivo: **reducir el MSE de ~0.25 a <0.01**
- El MSE es el "tablero de dardos" que guía el aprendizaje

En la siguiente sección veremos cómo **Backpropagation** usa este error para calcular exactamente cómo debe ajustar cada peso.

# 5. Backpropagation: Aprendiendo de los errores

Backpropagation es el **corazón del aprendizaje** en redes neuronales. Es el algoritmo que responde a la pregunta crucial: **"¿Cómo debe cambiar cada peso para reducir el error?"**

## Intuición: repartir la "culpa" del error hacia atrás

Imagina que tu equipo perdió un partido de fútbol por 3-0. ¿De quién es la culpa?

```python
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ===== IZQUIERDA: CULPA INJUSTA =====
ax1.axis('off')
ax1.text(0.5, 0.9, '❌ Enfoque Ingenuo', ha='center', fontsize=14,
        fontweight='bold', color='red', transform=ax1.transAxes)

roles = ['Portero', 'Defensa', 'Medio', 'Delantero']
blame = [1, 0, 0, 0]  # Solo el portero
colors_left = ['red', 'lightgray', 'lightgray', 'lightgray']

y_pos = 0.7
for role, b, color in zip(roles, blame, colors_left):
    ax1.text(0.5, y_pos, f'{role}: {b*100:.0f}% culpa',
            ha='center', fontsize=11, transform=ax1.transAxes,
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
    y_pos -= 0.15

ax1.text(0.5, 0.15, 'Injusto: todos contribuyeron\nal error en diferente medida',
        ha='center', fontsize=10, style='italic', transform=ax1.transAxes)

# ===== DERECHA: CULPA DISTRIBUIDA =====
ax2.axis('off')
ax2.text(0.5, 0.9, '✓ Backpropagation', ha='center', fontsize=14,
        fontweight='bold', color='green', transform=ax2.transAxes)

blame_distributed = [0.4, 0.3, 0.2, 0.1]  # Distribuida proporcionalmente
colors_right = ['lightcoral', 'orange', 'yellow', 'lightgreen']

y_pos = 0.7
for role, b, color in zip(roles, blame_distributed, colors_right):
    ax2.text(0.5, y_pos, f'{role}: {b*100:.0f}% culpa',
            ha='center', fontsize=11, transform=ax2.transAxes,
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
    y_pos -= 0.15

ax2.text(0.5, 0.15, 'Justo: cada uno recibe\nresponsabilidad según su contribución',
        ha='center', fontsize=10, style='italic', transform=ax2.transAxes)

plt.suptitle('Intuición: Distribuyendo la Responsabilidad del Error',
            fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

**Backpropagation hace exactamente esto con los pesos**: reparte la "culpa" del error proporcionalmente a cuánto contribuyó cada peso.

### El flujo hacia atrás

Mientras Forward Propagation va de **entrada → salida**, Backpropagation va de **salida → entrada**:

```python
fig, ax = plt.subplots(figsize=(12, 6))

# Positions
positions = [1, 4, 7, 10]
labels = ['Entrada\nX', 'Oculta\na⁽¹⁾', 'Salida\nŷ', 'Error\nMSE']
colors = ['lightblue', 'lightgreen', 'lightcoral', 'lightyellow']

# Draw forward flow (top)
for i in range(len(positions) - 1):
    ax.annotate('', xy=(positions[i+1], 4), xytext=(positions[i], 4),
               arrowprops=dict(arrowstyle='->', lw=3, color='blue'))
    ax.text((positions[i] + positions[i+1])/2, 4.3, 'Forward',
           ha='center', fontsize=10, color='blue', fontweight='bold')

# Draw backward flow (bottom)
for i in range(len(positions) - 1, 0, -1):
    ax.annotate('', xy=(positions[i-1], 2), xytext=(positions[i], 2),
               arrowprops=dict(arrowstyle='->', lw=3, color='red'))
    ax.text((positions[i] + positions[i-1])/2, 1.7, 'Backward',
           ha='center', fontsize=10, color='red', fontweight='bold')

# Draw boxes
for pos, label, color in zip(positions, labels, colors):
    rect = plt.Rectangle((pos-0.4, 1.5), 0.8, 3, facecolor=color,
                         edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(pos, 3, label, ha='center', va='center',
           fontsize=11, fontweight='bold')

# Add gradient labels
gradient_labels = ['∂MSE/∂W⁽²⁾\n"¿Cuánto cambiar\npesos finales?"',
                  '∂MSE/∂W⁽¹⁾\n"¿Cuánto cambiar\npesos iniciales?"']
gradient_pos = [8.5, 2.5]

for i, (label, pos) in enumerate(zip(gradient_labels, gradient_pos)):
    ax.text(pos, 0.8, label, ha='center', fontsize=9, style='italic',
           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.6))

ax.set_xlim(0, 11)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Flujo de Información: Forward vs. Backward',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```

## La Regla de la Cadena en palabras simples

Backpropagation se basa en una idea matemática llamada **Regla de la Cadena**. Suena complicado, pero es intuitivo.

### Analogía: El efecto dominó

Imagina que ajustas el volumen de tu TV:
1. **Giras la perilla** → Cambia señal eléctrica → Cambia amplificador → **Cambia volumen**

Si quieres saber "¿cuánto cambia el volumen cuando giro la perilla?", necesitas multiplicar cada paso:

```
Cambio_volumen = (Cambio_perilla) × (Efecto_señal) × (Efecto_amplificador)
```

**Esto es la Regla de la Cadena**: cuando hay una cadena de efectos, multiplicas las contribuciones de cada eslabón.

### Aplicado a nuestra red

```python
fig, ax = plt.subplots(figsize=(14, 7))

# Chain visualization
steps = [
    ('W⁽¹⁾', 'Peso entrada\n→ oculta', 0.5),
    ('→', '×', 1.5),
    ('z⁽¹⁾', 'Suma\nponderada', 2.5),
    ('→', '×', 3.5),
    ('a⁽¹⁾', 'Activación\n(sigmoide)', 4.5),
    ('→', '×', 5.5),
    ('z⁽²⁾', 'Suma\nponderada', 6.5),
    ('→', '×', 7.5),
    ('ŷ', 'Predicción\n(sigmoide)', 8.5),
    ('→', '×', 9.5),
    ('MSE', 'Error\nfinal', 10.5),
]

y_forward = 5
y_backward = 2

for i, (symbol, desc, x) in enumerate(steps):
    if symbol == '→':
        # Forward arrow
        prev_x = steps[i-1][2]
        next_x = steps[i+1][2]
        ax.annotate('', xy=(next_x-0.3, y_forward), xytext=(prev_x+0.3, y_forward),
                   arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
        ax.text(x, y_forward+0.3, 'afecta', ha='center', fontsize=8,
               color='blue', style='italic')
        
        # Backward arrow
        ax.annotate('', xy=(prev_x+0.3, y_backward), xytext=(next_x-0.3, y_backward),
                   arrowprops=dict(arrowstyle='->', lw=2, color='red'))
        
        # Multiplication symbol
        if symbol == '×' and i > 1:
            ax.text(x, y_backward-0.5, '×', ha='center', fontsize=16,
                   color='red', fontweight='bold')
    else:
        # Box
        color = 'lightyellow' if symbol == 'MSE' else 'lightblue' if 'W' in symbol else 'lightgreen'
        rect = plt.Rectangle((x-0.3, y_forward-0.4), 0.6, 0.8,
                            facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(x, y_forward, symbol, ha='center', va='center',
               fontsize=11, fontweight='bold')
        ax.text(x, y_forward-0.9, desc, ha='center', va='top',
               fontsize=8, style='italic')

# Labels
ax.text(5.5, y_forward+1.2, 'FORWARD: Cada paso afecta al siguiente',
       ha='center', fontsize=12, fontweight='bold', color='blue')
ax.text(5.5, y_backward-1.2, 'BACKWARD: Multiplicamos efectos (Regla de la Cadena)',
       ha='center', fontsize=12, fontweight='bold', color='red')

# Equation
ax.text(5.5, 0.5, '∂MSE/∂W⁽¹⁾ = (∂MSE/∂ŷ) × (∂ŷ/∂z⁽²⁾) × (∂z⁽²⁾/∂a⁽¹⁾) × (∂a⁽¹⁾/∂z⁽¹⁾) × (∂z⁽¹⁾/∂W⁽¹⁾)',
       ha='center', fontsize=10, style='italic',
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.set_xlim(0, 11.5)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('La Regla de la Cadena: Multiplicando Efectos',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```

**En palabras simples:** Para saber cuánto cambia el error cuando cambias W⁽¹⁾, multiplicas cuánto afecta W⁽¹⁾ a cada paso intermedio hasta llegar al error.

## Calculando gradientes (enfoque intuitivo)

Ahora calculemos los gradientes paso a paso, **sin matemáticas complicadas**:

### Paso 1: ¿Cuánto contribuye la salida al error?

```
∂MSE/∂ŷ = -(2/m) × (y_verdadero - ŷ)
```

**Intuición:** Si predijimos 0.8 pero debería ser 1, el gradiente es negativo → debemos **aumentar** ŷ.

### Paso 2: ¿Cuánto cambia la sigmoide?

```
∂ŷ/∂z⁽²⁾ = ŷ × (1 - ŷ)
```

**Intuición:** La sigmoide tiene más pendiente cerca de 0.5, menos en los extremos (0 o 1).

### Paso 3: Combinamos (Regla de la Cadena)

```
δ⁽²⁾ = (∂MSE/∂ŷ) × (∂ŷ/∂z⁽²⁾)
     = -(2/m) × (y - ŷ) × ŷ × (1 - ŷ)
```

Este **δ⁽²⁾** ("delta") es el error en la capa de salida.

### Implementación en código

```python
def backward_propagation(X, y_true, cache, W2):
    """
    Calcula gradientes usando backpropagation.
    
    Argumentos:
    X      -- entradas (m, 2)
    y_true -- etiquetas verdaderas (m, 1)
    cache  -- valores de forward propagation
    W2     -- pesos oculta→salida (4, 1)
    
    Retorna:
    gradients -- diccionario con dW1, db1, dW2, db2
    """
    m = X.shape[0]
    
    # Extraer valores del cache
    z1 = cache['z1']
    a1 = cache['a1']
    y_pred = cache['y_pred']
    
    # ==== PASO 1: Gradiente en la capa de salida ====
    # "¿Cuánto error hay en la salida?"
    delta2 = -(2/m) * (y_true - y_pred) * y_pred * (1 - y_pred)
    
    # Gradientes para W2 y b2
    dW2 = np.dot(a1.T, delta2)  # (4, m) × (m, 1) = (4, 1)
    db2 = np.sum(delta2, axis=0, keepdims=True)  # (1, 1)
    
    # ==== PASO 2: Propagar error a la capa oculta ====
    # "¿Cuánto de ese error es culpa de la capa oculta?"
    delta1 = np.dot(delta2, W2.T) * a1 * (1 - a1)
    
    # Gradientes para W1 y b1
    dW1 = np.dot(X.T, delta1)  # (2, m) × (m, 4) = (2, 4)
    db1 = np.sum(delta1, axis=0, keepdims=True)  # (1, 4)
    
    gradients = {
        'dW1': dW1,
        'db1': db1,
        'dW2': dW2,
        'db2': db2
    }
    
    return gradients

# Ejemplo con nuestros datos
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[0], [1], [1], [0]])

np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

# Forward propagation
z1 = np.dot(X_train, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
y_pred = sigmoid(z2)

cache = {'z1': z1, 'a1': a1, 'z2': z2, 'y_pred': y_pred}

# Backward propagation
gradients = backward_propagation(X_train, y_train, cache, W2)

print("="*70)
print("GRADIENTES CALCULADOS CON BACKPROPAGATION")
print("="*70)
print(f"\ndW2 (gradientes para pesos oculta→salida):")
print(f"Shape: {gradients['dW2'].shape}")
print(gradients['dW2'])
print(f"\nInterpretación: Cada valor indica cuánto debe cambiar ese peso")
print(f"  - Valor negativo → aumentar el peso")
print(f"  - Valor positivo → disminuir el peso")
print(f"  - Magnitud grande → cambio importante necesario")

print(f"\ndW1 (gradientes para pesos entrada→oculta):")
print(f"Shape: {gradients['dW1'].shape}")
print(gradients['dW1'])

print(f"\ndb1 (gradientes para sesgos oculta): {gradients['db1']}")
print(f"db2 (gradiente para sesgo salida): {gradients['db2'][0,0]:.6f}")
print("="*70)
```

### Visualizando los gradientes

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== IZQUIERDA: GRADIENTES W2 =====
dW2 = gradients['dW2'].flatten()
colors = ['green' if g < 0 else 'red' for g in dW2]

ax1.barh(range(len(dW2)), dW2, color=colors, alpha=0.7,
        edgecolor='black', linewidth=2)
ax1.set_yticks(range(len(dW2)))
ax1.set_yticklabels([f'w{i+1}→ŷ' for i in range(len(dW2))])
ax1.set_xlabel('Magnitud del Gradiente', fontsize=11, fontweight='bold')
ax1.set_title('Gradientes W⁽²⁾\n(Oculta → Salida)', fontsize=12, fontweight='bold')
ax1.axvline(x=0, color='black', linewidth=1)
ax1.grid(axis='x', alpha=0.3)

# Annotations
ax1.text(0.5, len(dW2)+0.5, '← Aumentar peso', ha='center', fontsize=9,
        color='green', fontweight='bold')
ax1.text(-0.5, len(dW2)+0.5, 'Disminuir peso →', ha='center', fontsize=9,
        color='red', fontweight='bold')

# ===== DERECHA: GRADIENTES W1 =====
dW1 = gradients['dW1']
im = ax2.imshow(dW1.T, cmap='RdYlGn_r', aspect='auto', vmin=-1, vmax=1)
ax2.set_xticks(range(2))
ax2.set_yticks(range(4))
ax2.set_xticklabels(['x₁', 'x₂'])
ax2.set_yticklabels([f'h{i+1}' for i in range(4)])
ax2.set_title('Gradientes W⁽¹⁾\n(Entrada → Oculta)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax2, label='Magnitud (rojo=↓, verde=↑)')

for i in range(4):
    for j in range(2):
        ax2.text(j, i, f'{dW1[j, i]:.2f}', ha='center', va='center',
                fontweight='bold', fontsize=9)

plt.suptitle('Visualización de Gradientes: ¿Cómo Ajustar Cada Peso?',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```

---

**Recapitulación:**
- **Backpropagation** = distribuir la "culpa" del error hacia atrás
- **Regla de la Cadena** = multiplicar efectos a través de la cadena
- Los **gradientes** indican dirección y magnitud del ajuste necesario
- Negativo → **aumentar** peso, Positivo → **disminuir** peso
- El código es sorprendentemente simple (~15 líneas)

En la siguiente sección veremos cómo **usar estos gradientes** para actualizar los pesos y hacer que la red aprenda.

# 6. Descenso del Gradiente: Mejorando los pesos

Ya sabemos **qué tan mal** lo hace la red (función de coste) y **en qué dirección** ajustar cada peso (backpropagation). Ahora viene la parte más emocionante: **actualizar los pesos para mejorar**.

## Actualizando pesos en la dirección correcta

El Descenso del Gradiente es sorprendentemente simple. Es como bajar una montaña en la niebla:

```python
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 6))

# ===== IZQUIERDA: ANALOGÍA DE LA MONTAÑA =====
ax1 = fig.add_subplot(121)

# Simulate mountain descent
x = np.linspace(-3, 3, 100)
y = x**2 + 1  # Simple parabola (valley)

# Gradient descent path
path_x = [2.5, 2.0, 1.3, 0.7, 0.2, 0.0]
path_y = [px**2 + 1 for px in path_x]

ax1.plot(x, y, 'b-', linewidth=2, label='Superficie de error')
ax1.plot(path_x, path_y, 'ro-', linewidth=2, markersize=10,
        label='Descenso del gradiente')

# Annotations
ax1.annotate('Inicio\n(pesos aleatorios)', xy=(path_x[0], path_y[0]),
            xytext=(path_x[0]+0.5, path_y[0]+2),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, fontweight='bold', color='red')
ax1.annotate('Mínimo\n(pesos óptimos)', xy=(path_x[-1], path_y[-1]),
            xytext=(path_x[-1]+0.8, path_y[-1]+2),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=10, fontweight='bold', color='green')

# Gradient arrows
for i in range(len(path_x)-1):
    ax1.annotate('', xy=(path_x[i+1], path_y[i+1]),
                xytext=(path_x[i], path_y[i]),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2, alpha=0.5))

ax1.set_xlabel('Peso', fontsize=11, fontweight='bold')
ax1.set_ylabel('Error (MSE)', fontsize=11, fontweight='bold')
ax1.set_title('Analogía: Bajando la Montaña', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ===== DERECHA: LA REGLA DE ACTUALIZACIÓN =====
ax2 = fig.add_subplot(122)
ax2.axis('off')

ax2.text(0.5, 0.95, 'Regla de Actualización', ha='center', fontsize=14,
        fontweight='bold', transform=ax2.transAxes)

# Main equation
ax2.text(0.5, 0.75, 'θ_nuevo = θ_viejo - α × gradiente',
        ha='center', fontsize=16, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7),
        transform=ax2.transAxes, family='monospace')

# Explanation boxes
explanations = [
    ('θ_viejo', 'Peso actual', 0.55, 'lightblue'),
    ('α (alpha)', 'Learning Rate\n(tamaño del paso)', 0.40, 'lightgreen'),
    ('gradiente', '¿Hacia dónde ir?\n(de backpropagation)', 0.25, 'lightcoral'),
    ('θ_nuevo', 'Peso actualizado', 0.10, 'lightyellow'),
]

for symbol, desc, y_pos, color in explanations:
    ax2.text(0.5, y_pos, f'{symbol}\n{desc}', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.7),
            transform=ax2.transAxes)

plt.suptitle('Descenso del Gradiente: Actualización Iterativa de Pesos',
            fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

### La regla en código

```python
def update_parameters(W1, b1, W2, b2, gradients, learning_rate):
    """
    Actualiza pesos usando descenso del gradiente.
    
    Argumentos:
    W1, b1, W2, b2 -- parámetros actuales
    gradients      -- gradientes de backpropagation
    learning_rate  -- tamaño del paso (α)
    
    Retorna:
    W1, b1, W2, b2 -- parámetros actualizados
    """
    # Actualizar cada parámetro
    W1 = W1 - learning_rate * gradients['dW1']
    b1 = b1 - learning_rate * gradients['db1']
    W2 = W2 - learning_rate * gradients['dW2']
    b2 = b2 - learning_rate * gradients['db2']
    
    return W1, b1, W2, b2

# Ejemplo: una iteración de actualización
print("="*70)
print("EJEMPLO: UNA ITERACIÓN DE DESCENSO DEL GRADIENTE")
print("="*70)

# Parámetros iniciales (de secciones anteriores)
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
W2 = np.random.randn(4, 1) * 0.5

print("\nPeso W2[0,0] ANTES de actualizar:")
print(f"  Valor:     {W2[0, 0]:.6f}")
print(f"  Gradiente: {gradients['dW2'][0, 0]:.6f}")

# Actualizar con learning rate = 0.5
learning_rate = 0.5
W1_new, b1_new, W2_new, b2_new = update_parameters(W1, b1, W2, b2,
                                                     gradients, learning_rate)

print(f"\nPeso W2[0,0] DESPUÉS de actualizar:")
print(f"  Valor:     {W2_new[0, 0]:.6f}")
print(f"  Cambio:    {W2_new[0, 0] - W2[0, 0]:.6f}")
print(f"  (= -α × gradiente = -{learning_rate} × {gradients['dW2'][0, 0]:.6f})")
print("="*70)
```

## El Learning Rate: ¿pasos grandes o pequeños?

El **learning rate** (α) es uno de los hiperparámetros más importantes. Controla **qué tan grandes son los pasos** que damos hacia el mínimo.

```python
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Función de ejemplo (parábola simple)
x = np.linspace(-5, 5, 200)
y = x**2

# ===== SUBPLOT 1: LEARNING RATE ÓPTIMO =====
ax1 = axes[0, 0]
path_good = [4, 2.8, 1.6, 0.8, 0.2, 0.0]
ax1.plot(x, y, 'b-', linewidth=2, alpha=0.5)
ax1.plot(path_good, [p**2 for p in path_good], 'go-',
        linewidth=2, markersize=10, label='α = 0.3 (óptimo)')
ax1.set_xlabel('Peso', fontsize=10, fontweight='bold')
ax1.set_ylabel('Error', fontsize=10, fontweight='bold')
ax1.set_title('✓ Learning Rate Óptimo\nConvergencia rápida y estable',
             fontsize=11, fontweight='bold', color='green')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-1, 20)

# ===== SUBPLOT 2: LEARNING RATE MUY PEQUEÑO =====
ax2 = axes[0, 1]
path_slow = np.linspace(4, 1, 15)
ax2.plot(x, y, 'b-', linewidth=2, alpha=0.5)
ax2.plot(path_slow, path_slow**2, 'yo-', linewidth=2,
        markersize=8, label='α = 0.01 (muy pequeño)')
ax2.set_xlabel('Peso', fontsize=10, fontweight='bold')
ax2.set_ylabel('Error', fontsize=10, fontweight='bold')
ax2.set_title('⚠️ Learning Rate Muy Pequeño\nMuy lento, muchas iteraciones',
             fontsize=11, fontweight='bold', color='orange')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-1, 20)

# ===== SUBPLOT 3: LEARNING RATE MUY GRANDE =====
ax3 = axes[1, 0]
path_oscillate = [4, -3, 2.5, -2, 1.5, -1, 0.8, -0.5, 0.3]
ax3.plot(x, y, 'b-', linewidth=2, alpha=0.5)
ax3.plot(path_oscillate, [p**2 for p in path_oscillate], 'ro-',
        linewidth=2, markersize=8, label='α = 1.8 (muy grande)')
ax3.set_xlabel('Peso', fontsize=10, fontweight='bold')
ax3.set_ylabel('Error', fontsize=10, fontweight='bold')
ax3.set_title('❌ Learning Rate Muy Grande\nOscila, no converge',
             fontsize=11, fontweight='bold', color='red')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_ylim(-1, 20)

# ===== SUBPLOT 4: COMPARACIÓN DE CONVERGENCIA =====
ax4 = axes[1, 1]

# Simulate training with different learning rates
def simulate_training(lr, epochs=50):
    w = 4.0  # start
    history = [w**2]
    for _ in range(epochs):
        grad = 2 * w  # derivative of x^2
        w = w - lr * grad
        history.append(w**2)
        if abs(w) < 0.01:
            break
    return history

epochs_range = 50
loss_optimal = simulate_training(0.3, epochs_range)
loss_slow = simulate_training(0.01, epochs_range)
loss_fast = simulate_training(1.8, epochs_range)[:epochs_range+1]  # Limit length

ax4.plot(range(len(loss_optimal)), loss_optimal, 'g-', linewidth=2,
        label='α = 0.3 (óptimo)', marker='o', markersize=4)
ax4.plot(range(len(loss_slow)), loss_slow, 'y-', linewidth=2,
        label='α = 0.01 (lento)', marker='s', markersize=4)
ax4.plot(range(len(loss_fast)), loss_fast, 'r-', linewidth=2,
        label='α = 1.8 (oscila)', marker='^', markersize=4)

ax4.axhline(y=0.01, color='green', linestyle='--', linewidth=1.5,
           label='Objetivo', alpha=0.5)
ax4.set_xlabel('Época', fontsize=10, fontweight='bold')
ax4.set_ylabel('Error (log scale)', fontsize=10, fontweight='bold')
ax4.set_title('Comparación de Convergencia', fontsize=11, fontweight='bold')
ax4.set_yscale('log')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3, which='both')

plt.suptitle('Impacto del Learning Rate en el Entrenamiento',
            fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

### ¿Cómo elegir el learning rate?

```python
print("\n" + "="*70)
print("GUÍA PARA ELEGIR EL LEARNING RATE")
print("="*70)

lr_guide = {
    "α = 0.001 - 0.01": "Muy conservador, para problemas complejos",
    "α = 0.1 - 0.5":    "Rango típico para XOR y problemas simples ✓",
    "α = 1.0 - 2.0":    "Agresivo, puede oscilar en algunos casos",
    "α > 5.0":          "Probablemente divergirá ❌",
}

for lr_range, description in lr_guide.items():
    print(f"  {lr_range:<20} → {description}")

print("\n" + "-"*70)
print("Para nuestro problema XOR usaremos: α = 0.5")
print("="*70)
```

## Visualización de la convergencia

Ahora juntemos todo: **entrenamiento completo** con múltiples iteraciones.

```python
def train_network(X, y, learning_rate=0.5, epochs=1000):
    """
    Entrena la red neuronal usando descenso del gradiente.
    
    Argumentos:
    X             -- datos de entrada (4, 2)
    y             -- etiquetas verdaderas (4, 1)
    learning_rate -- tamaño del paso
    epochs        -- número de iteraciones
    
    Retorna:
    W1, b1, W2, b2 -- parámetros entrenados
    history        -- historial de MSE
    """
    # Inicializar parámetros
    np.random.seed(42)
    W1 = np.random.randn(2, 4) * 0.5
    b1 = np.zeros((1, 4))
    W2 = np.random.randn(4, 1) * 0.5
    b2 = np.zeros((1, 1))
    
    history = []
    
    for epoch in range(epochs):
        # Forward propagation
        z1 = np.dot(X, W1) + b1
        a1 = sigmoid(z1)
        z2 = np.dot(a1, W2) + b2
        y_pred = sigmoid(z2)
        
        # Calcular error
        mse = compute_mse(y, y_pred)
        history.append(mse)
        
        # Backward propagation
        cache = {'z1': z1, 'a1': a1, 'z2': z2, 'y_pred': y_pred}
        gradients = backward_propagation(X, y, cache, W2)
        
        # Actualizar parámetros (DESCENSO DEL GRADIENTE)
        W1 = W1 - learning_rate * gradients['dW1']
        b1 = b1 - learning_rate * gradients['db1']
        W2 = W2 - learning_rate * gradients['dW2']
        b2 = b2 - learning_rate * gradients['db2']
        
        # Imprimir progreso cada 200 épocas
        if epoch % 200 == 0 or epoch == epochs - 1:
            print(f"Época {epoch:4d} | MSE: {mse:.6f}")
    
    return W1, b1, W2, b2, history

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def compute_mse(y_true, y_pred):
    return (1 / y_true.shape[0]) * np.sum((y_true - y_pred) ** 2)

def backward_propagation(X, y_true, cache, W2):
    m = X.shape[0]
    a1 = cache['a1']
    y_pred = cache['y_pred']
    
    delta2 = -(2/m) * (y_true - y_pred) * y_pred * (1 - y_pred)
    dW2 = np.dot(a1.T, delta2)
    db2 = np.sum(delta2, axis=0, keepdims=True)
    
    delta1 = np.dot(delta2, W2.T) * a1 * (1 - a1)
    dW1 = np.dot(X.T, delta1)
    db1 = np.sum(delta1, axis=0, keepdims=True)
    
    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

# Entrenar la red
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[0], [1], [1], [0]])

print("="*70)
print("ENTRENANDO LA RED NEURONAL")
print("="*70)
W1, b1, W2, b2, history = train_network(X_train, y_train,
                                         learning_rate=0.5, epochs=1000)
print("="*70)

# Visualizar convergencia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== IZQUIERDA: CURVA DE PÉRDIDA (ESCALA LINEAR) =====
ax1.plot(history, linewidth=2, color='blue', alpha=0.7)
ax1.axhline(y=0.01, color='green', linestyle='--', linewidth=2,
           label='Objetivo (MSE < 0.01)', alpha=0.7)
ax1.fill_between(range(len(history)), 0, history, alpha=0.2, color='blue')

# Anotar puntos clave
milestones = [0, 100, 500, 999]
for m in milestones:
    ax1.plot(m, history[m], 'ro', markersize=8)
    ax1.annotate(f'{history[m]:.4f}', xy=(m, history[m]),
                xytext=(m, history[m] + 0.02),
                ha='center', fontsize=9, fontweight='bold')

ax1.set_xlabel('Época', fontsize=11, fontweight='bold')
ax1.set_ylabel('MSE', fontsize=11, fontweight='bold')
ax1.set_title('Curva de Pérdida (Escala Lineal)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ===== DERECHA: CURVA DE PÉRDIDA (ESCALA LOG) =====
ax2.semilogy(history, linewidth=2, color='blue', alpha=0.7)
ax2.axhline(y=0.01, color='green', linestyle='--', linewidth=2,
           label='Objetivo (MSE < 0.01)', alpha=0.7)

ax2.set_xlabel('Época', fontsize=11, fontweight='bold')
ax2.set_ylabel('MSE (escala log)', fontsize=11, fontweight='bold')
ax2.set_title('Curva de Pérdida (Escala Logarítmica)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which='both')

plt.suptitle('Convergencia del Entrenamiento: Descenso del Gradiente en Acción',
            fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✓ ¡Entrenamiento completado!")
print(f"  MSE inicial: {history[0]:.6f}")
print(f"  MSE final:   {history[-1]:.6f}")
print(f"  Reducción:   {(1 - history[-1]/history[0]) * 100:.1f}%")
```

---

**Recapitulación:**
- **Descenso del Gradiente** = actualizar pesos en dirección opuesta al gradiente
- Regla simple: `θ_nuevo = θ_viejo - α × gradiente`
- **Learning rate (α)** controla el tamaño del paso
  - Muy pequeño → lento
  - Óptimo (0.1-0.5 para XOR) → convergencia rápida
  - Muy grande → oscila o diverge
- La **curva de pérdida** muestra el progreso del aprendizaje
- ¡Nuestra red está aprendiendo XOR! 🎉

En la siguiente sección veremos el **código completo integrado** y los resultados finales.

# 7. Implementación Completa: Código desde Cero

Ha llegado el momento de juntar todas las piezas. Aquí está la implementación completa de nuestra red neuronal en una clase limpia y bien organizada.

## Clase NeuralNetwork completa en ~100 líneas

```python
import numpy as np
import matplotlib.pyplot as plt

class NeuralNetwork:
    """
    Red Neuronal Multicapa (MLP) para clasificación binaria.
    Arquitectura: 2 entradas → 4 neuronas ocultas → 1 salida
    """
    
    def __init__(self, learning_rate=0.5, seed=42):
        """
        Inicializa la red neuronal.
        
        Argumentos:
        learning_rate -- tasa de aprendizaje (α)
        seed          -- semilla para reproducibilidad
        """
        self.learning_rate = learning_rate
        
        # Inicializar pesos y sesgos aleatoriamente
        np.random.seed(seed)
        self.W1 = np.random.randn(2, 4) * 0.5  # Entrada → Oculta
        self.b1 = np.zeros((1, 4))
        self.W2 = np.random.randn(4, 1) * 0.5  # Oculta → Salida
        self.b2 = np.zeros((1, 1))
        
        # Historial de entrenamiento
        self.history = []
    
    def sigmoid(self, z):
        """Función de activación sigmoide."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def forward(self, X):
        """
        Forward propagation: calcula predicciones.
        
        Argumentos:
        X -- datos de entrada (m, 2)
        
        Retorna:
        y_pred -- predicciones (m, 1)
        cache  -- valores intermedios
        """
        # Capa oculta
        z1 = np.dot(X, self.W1) + self.b1
        a1 = self.sigmoid(z1)
        
        # Capa de salida
        z2 = np.dot(a1, self.W2) + self.b2
        y_pred = self.sigmoid(z2)
        
        # Guardar para backpropagation
        cache = {'z1': z1, 'a1': a1, 'z2': z2, 'y_pred': y_pred}
        return y_pred, cache
    
    def compute_cost(self, y_true, y_pred):
        """Calcula el Error Cuadrático Medio (MSE)."""
        m = y_true.shape[0]
        return (1 / m) * np.sum((y_true - y_pred) ** 2)
    
    def backward(self, X, y_true, cache):
        """
        Backpropagation: calcula gradientes.
        
        Argumentos:
        X      -- datos de entrada (m, 2)
        y_true -- etiquetas verdaderas (m, 1)
        cache  -- valores de forward propagation
        
        Retorna:
        gradients -- diccionario con dW1, db1, dW2, db2
        """
        m = X.shape[0]
        a1 = cache['a1']
        y_pred = cache['y_pred']
        
        # Gradientes capa de salida
        delta2 = -(2/m) * (y_true - y_pred) * y_pred * (1 - y_pred)
        dW2 = np.dot(a1.T, delta2)
        db2 = np.sum(delta2, axis=0, keepdims=True)
        
        # Gradientes capa oculta
        delta1 = np.dot(delta2, self.W2.T) * a1 * (1 - a1)
        dW1 = np.dot(X.T, delta1)
        db1 = np.sum(delta1, axis=0, keepdims=True)
        
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    
    def update_parameters(self, gradients):
        """Actualiza pesos usando descenso del gradiente."""
        self.W1 -= self.learning_rate * gradients['dW1']
        self.b1 -= self.learning_rate * gradients['db1']
        self.W2 -= self.learning_rate * gradients['dW2']
        self.b2 -= self.learning_rate * gradients['db2']
    
    def train(self, X, y, epochs=1000, verbose=True):
        """
        Entrena la red neuronal.
        
        Argumentos:
        X       -- datos de entrada (m, 2)
        y       -- etiquetas (m, 1)
        epochs  -- número de iteraciones
        verbose -- si imprimir progreso
        """
        if verbose:
            print("="*60)
            print("ENTRENAMIENTO INICIADO")
            print("="*60)
        
        for epoch in range(epochs):
            # Forward propagation
            y_pred, cache = self.forward(X)
            
            # Calcular error
            cost = self.compute_cost(y, y_pred)
            self.history.append(cost)
            
            # Backpropagation
            gradients = self.backward(X, y, cache)
            
            # Actualizar pesos
            self.update_parameters(gradients)
            
            # Mostrar progreso
            if verbose and (epoch % 200 == 0 or epoch == epochs - 1):
                print(f"Época {epoch:4d} | MSE: {cost:.8f}")
        
        if verbose:
            print("="*60)
            print("✓ ENTRENAMIENTO COMPLETADO")
            print("="*60)
    
    def predict(self, X, threshold=0.5):
        """
        Hace predicciones.
        
        Argumentos:
        X         -- datos de entrada (m, 2)
        threshold -- umbral de decisión
        
        Retorna:
        predictions   -- clases predichas (m, 1)
        probabilities -- probabilidades (m, 1)
        """
        probabilities, _ = self.forward(X)
        predictions = (probabilities >= threshold).astype(int)
        return predictions, probabilities

# ¡Solo ~100 líneas de código para una red neuronal completa!
print("✓ Clase NeuralNetwork definida")
print(f"  Líneas de código: ~100")
print(f"  Métodos principales: __init__, forward, backward, train, predict")
```

## Entrenamiento y visualización de resultados

Ahora usemos nuestra clase para resolver XOR:

```python
# Dataset XOR
X_train = np.array([[0, 0],
                    [0, 1],
                    [1, 0],
                    [1, 1]])

y_train = np.array([[0],
                    [1],
                    [1],
                    [0]])

# Crear y entrenar la red
nn = NeuralNetwork(learning_rate=0.5, seed=42)
nn.train(X_train, y_train, epochs=2000, verbose=True)

# Hacer predicciones
predictions, probabilities = nn.predict(X_train)

# Mostrar resultados
print("\n" + "="*60)
print("RESULTADOS DEL ENTRENAMIENTO")
print("="*60)
print(f"\n{'Entrada':<12} {'Verdadero':<12} {'Probabilidad':<15} {'Predicción':<12} {'Estado'}")
print("-"*60)

for i in range(len(X_train)):
    x = X_train[i]
    true = y_train[i, 0]
    prob = probabilities[i, 0]
    pred = predictions[i, 0]
    status = "✓ Correcto" if pred == true else "✗ Error"
    
    print(f"{str(tuple(x)):<12} {true:<12} {prob:<15.6f} {pred:<12} {status}")

# Calcular precisión
accuracy = np.mean(predictions == y_train) * 100
print("-"*60)
print(f"Precisión: {accuracy:.1f}%")
print("="*60)
```

## Curva de pérdida y predicciones finales

Visualicemos todo el proceso de aprendizaje:

```python
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

# ===== SUBPLOT 1: CURVA DE PÉRDIDA (LINEAL) =====
ax1 = fig.add_subplot(gs[0, 0])

ax1.plot(nn.history, linewidth=2, color='blue', alpha=0.7, label='MSE')
ax1.axhline(y=0.01, color='green', linestyle='--', linewidth=2,
           label='Objetivo (MSE < 0.01)', alpha=0.7)
ax1.fill_between(range(len(nn.history)), 0, nn.history, alpha=0.2, color='blue')

# Anotar convergencia
converged_epoch = next((i for i, cost in enumerate(nn.history) if cost < 0.01), None)
if converged_epoch:
    ax1.plot(converged_epoch, nn.history[converged_epoch], 'go', markersize=12)
    ax1.annotate(f'Convergió en\népoca {converged_epoch}',
                xy=(converged_epoch, nn.history[converged_epoch]),
                xytext=(converged_epoch + 200, 0.05),
                arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=10, fontweight='bold', color='green')

ax1.set_xlabel('Época', fontsize=11, fontweight='bold')
ax1.set_ylabel('MSE', fontsize=11, fontweight='bold')
ax1.set_title('Curva de Pérdida (Escala Lineal)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ===== SUBPLOT 2: CURVA DE PÉRDIDA (LOG) =====
ax2 = fig.add_subplot(gs[0, 1])

ax2.semilogy(nn.history, linewidth=2, color='blue', alpha=0.7, label='MSE')
ax2.axhline(y=0.01, color='green', linestyle='--', linewidth=2,
           label='Objetivo', alpha=0.7)

ax2.set_xlabel('Época', fontsize=11, fontweight='bold')
ax2.set_ylabel('MSE (escala log)', fontsize=11, fontweight='bold')
ax2.set_title('Curva de Pérdida (Escala Logarítmica)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which='both')

# ===== SUBPLOT 3: PREDICCIONES =====
ax3 = fig.add_subplot(gs[1, 0])

input_labels = [f'{tuple(x)}→{y[0]}' for x, y in zip(X_train, y_train)]
colors = ['green' if pred[0] == true[0] else 'red'
          for pred, true in zip(predictions, y_train)]

bars = ax3.bar(input_labels, probabilities.flatten(), color=colors,
              alpha=0.7, edgecolor='black', linewidth=2)
ax3.axhline(y=0.5, color='blue', linestyle='--', linewidth=2,
           label='Umbral (0.5)', alpha=0.6)

for i, (bar, prob, pred, true) in enumerate(zip(bars, probabilities.flatten(),
                                                 predictions.flatten(), y_train.flatten())):
    height = bar.get_height()
    status = '✓' if pred == true else '✗'
    ax3.text(i, height + 0.05, f'{prob:.4f}\n→ {pred} {status}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax3.set_xlabel('Entrada → Verdadero', fontsize=11, fontweight='bold')
ax3.set_ylabel('Probabilidad Predicha', fontsize=11, fontweight='bold')
ax3.set_title('Predicciones Finales', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(0, 1.15)

# ===== SUBPLOT 4: FRONTERA DE DECISIÓN =====
ax4 = fig.add_subplot(gs[1, 1])

# Crear malla
h = 0.01
x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predecir en la malla
grid_points = np.c_[xx.ravel(), yy.ravel()]
_, Z = nn.predict(grid_points)
Z = Z.reshape(xx.shape)

# Plotear
contourf = ax4.contourf(xx, yy, Z, levels=20, cmap='RdYlBu_r', alpha=0.8)
ax4.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)
plt.colorbar(contourf, ax=ax4, label='P(y=1)')

# Plotear puntos de entrenamiento
for i, (x, y) in enumerate(zip(X_train, y_train)):
    color = 'blue' if y[0] == 1 else 'red'
    marker = 's' if y[0] == 1 else 'o'
    ax4.scatter(x[0], x[1], c=color, marker=marker, s=300,
               edgecolors='black', linewidths=2, zorder=10)
    ax4.annotate(f'{tuple(x)}', xy=x, xytext=(5, 5),
                textcoords='offset points', fontsize=9, fontweight='bold')

ax4.set_xlabel('x₁', fontsize=11, fontweight='bold')
ax4.set_ylabel('x₂', fontsize=11, fontweight='bold')
ax4.set_title('Frontera de Decisión Aprendida', fontsize=12, fontweight='bold')
ax4.set_xlim(x_min, x_max)
ax4.set_ylim(y_min, y_max)
ax4.grid(True, alpha=0.3)

# ===== SUBPLOT 5: EVOLUCIÓN DE PREDICCIONES =====
ax5 = fig.add_subplot(gs[2, :])

# Simular predicciones en diferentes épocas
epochs_to_show = [0, 100, 500, 1000, 2000]
predictions_evolution = []

for target_epoch in epochs_to_show:
    # Entrenar hasta esa época
    nn_temp = NeuralNetwork(learning_rate=0.5, seed=42)
    if target_epoch > 0:
        nn_temp.train(X_train, y_train, epochs=target_epoch, verbose=False)
    _, probs = nn_temp.predict(X_train)
    predictions_evolution.append(probs.flatten())

x_pos = np.arange(len(X_train))
width = 0.15

for i, (epoch, preds) in enumerate(zip(epochs_to_show, predictions_evolution)):
    offset = width * (i - 2)
    colors = ['green' if abs(pred - true) < 0.1 else 'orange' if abs(pred - true) < 0.3 else 'red'
              for pred, true in zip(preds, y_train.flatten())]
    ax5.bar(x_pos + offset, preds, width, label=f'Época {epoch}', alpha=0.7)

ax5.axhline(y=0.5, color='blue', linestyle='--', linewidth=2, alpha=0.5)
ax5.set_xticks(x_pos)
ax5.set_xticklabels(input_labels, fontsize=10)
ax5.set_xlabel('Entrada → Verdadero', fontsize=11, fontweight='bold')
ax5.set_ylabel('Probabilidad Predicha', fontsize=11, fontweight='bold')
ax5.set_title('Evolución de las Predicciones Durante el Entrenamiento',
             fontsize=12, fontweight='bold')
ax5.legend(fontsize=9, ncol=5, loc='upper center')
ax5.grid(axis='y', alpha=0.3)
ax5.set_ylim(0, 1.1)

plt.suptitle('Red Neuronal Entrenada: Análisis Completo',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
```

### Estadísticas finales

```python
print("\n" + "="*60)
print("ESTADÍSTICAS FINALES DEL ENTRENAMIENTO")
print("="*60)

print(f"\nArquitectura:")
print(f"  • Entrada:      2 neuronas")
print(f"  • Capa oculta:  4 neuronas")
print(f"  • Salida:       1 neurona")
print(f"  • Total params: 17 parámetros")

print(f"\nHiperparámetros:")
print(f"  • Learning rate: {nn.learning_rate}")
print(f"  • Épocas:        {len(nn.history)}")

print(f"\nRendimiento:")
print(f"  • MSE inicial:  {nn.history[0]:.6f}")
print(f"  • MSE final:    {nn.history[-1]:.6f}")
print(f"  • Reducción:    {(1 - nn.history[-1]/nn.history[0])*100:.2f}%")
print(f"  • Precisión:    {accuracy:.1f}%")

if converged_epoch:
    print(f"\nConvergencia:")
    print(f"  • Convergió en época: {converged_epoch}")
    print(f"  • MSE en convergencia: {nn.history[converged_epoch]:.6f}")

print("\n" + "="*60)
print("✓ ¡La red ha aprendido a resolver XOR perfectamente!")
print("="*60)
```

### Comparación: antes vs después del entrenamiento

```python
# Crear red sin entrenar
nn_untrained = NeuralNetwork(learning_rate=0.5, seed=42)
_, probs_before = nn_untrained.predict(X_train)

# Comparar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

input_labels = [f'{tuple(x)}' for x in X_train]

# ANTES del entrenamiento
ax1.bar(input_labels, probs_before.flatten(), color='red',
       alpha=0.7, edgecolor='black', linewidth=2)
ax1.axhline(y=0.5, color='blue', linestyle='--', linewidth=2, alpha=0.6)
ax1.set_xlabel('Entrada', fontsize=11, fontweight='bold')
ax1.set_ylabel('Probabilidad Predicha', fontsize=11, fontweight='bold')
ax1.set_title('ANTES del Entrenamiento\n(Pesos Aleatorios)',
             fontsize=12, fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.grid(axis='y', alpha=0.3)

for i, prob in enumerate(probs_before.flatten()):
    ax1.text(i, prob + 0.05, f'{prob:.3f}', ha='center',
            fontweight='bold', fontsize=10)

# DESPUÉS del entrenamiento
colors = ['green' if pred[0] == true[0] else 'red'
          for pred, true in zip(predictions, y_train)]
ax2.bar(input_labels, probabilities.flatten(), color=colors,
       alpha=0.7, edgecolor='black', linewidth=2)
ax2.axhline(y=0.5, color='blue', linestyle='--', linewidth=2, alpha=0.6)
ax2.set_xlabel('Entrada', fontsize=11, fontweight='bold')
ax2.set_ylabel('Probabilidad Predicha', fontsize=11, fontweight='bold')
ax2.set_title('DESPUÉS del Entrenamiento\n(Pesos Optimizados)',
             fontsize=12, fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.grid(axis='y', alpha=0.3)

for i, (prob, true) in enumerate(zip(probabilities.flatten(), y_train.flatten())):
    ax2.text(i, prob + 0.05, f'{prob:.3f}\n(real: {true})',
            ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Transformación: De Pesos Aleatorios a Solución Perfecta',
            fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
```

---

**Recapitulación:**
- ✅ **Implementación completa en ~100 líneas**: limpia, modular, reutilizable
- ✅ **Entrenamiento exitoso**: MSE < 0.01, precisión 100%
- ✅ **La red aprende por sí misma**: de pesos aleatorios a solución perfecta
- ✅ **Visualizaciones completas**: curva de pérdida, predicciones, frontera de decisión
- ✅ **XOR resuelto**: el problema "imposible" para un perceptrón simple

En la siguiente sección veremos cómo hacer lo mismo con scikit-learn en solo ~10 líneas de código.

# 8. La Vía Rápida: Scikit-Learn

Hemos invertido mucho esfuerzo en entender cómo funciona una red neuronal desde cero. Ahora veamos cómo resolver el mismo problema usando **scikit-learn**, una de las librerías de Machine Learning más populares.

## Resolviendo XOR en 10 líneas

```python
from sklearn.neural_network import MLPClassifier
import numpy as np

# Dataset XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

# Crear y entrenar la red (¡solo 3 líneas!)
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='logistic',
                    solver='sgd', learning_rate_init=0.5, max_iter=2000,
                    random_state=42)
mlp.fit(X, y)

# Hacer predicciones
predictions = mlp.predict(X)
probabilities = mlp.predict_proba(X)[:, 1]

# Mostrar resultados
accuracy = (predictions == y).mean() * 100

print("="*60)
print("SCIKIT-LEARN: XOR RESUELTO EN 10 LÍNEAS")
print("="*60)
print(f"\nPrecisión: {accuracy:.1f}%")
print(f"Iteraciones: {mlp.n_iter_}")
print(f"Pérdida final: {mlp.loss_:.6f}")
print("\nPredicciones:")
for i, (x_val, true, pred, prob) in enumerate(zip(X, y, predictions, probabilities)):
    status = "✓" if pred == true else "✗"
    print(f"  {tuple(x_val)} → Verdadero: {true}, Predicho: {pred} ({prob:.4f}) {status}")
print("="*60)
```

**¡Eso es todo!** En 10 líneas hemos resuelto XOR con la misma arquitectura que nos tomó ~100 líneas implementar.

### Desglosando los parámetros

```python
print("\n" + "="*60)
print("PARÁMETROS DE MLPClassifier")
print("="*60)

params_explanation = {
    'hidden_layer_sizes': ('(4,)', 'Una capa oculta con 4 neuronas'),
    'activation': ('logistic', 'Función sigmoide (igual que la nuestra)'),
    'solver': ('sgd', 'Stochastic Gradient Descent'),
    'learning_rate_init': ('0.5', 'Learning rate (α)'),
    'max_iter': ('2000', 'Número máximo de épocas'),
    'random_state': ('42', 'Semilla para reproducibilidad'),
}

for param, (value, description) in params_explanation.items():
    print(f"\n{param}:")
    print(f"  Valor:       {value}")
    print(f"  Descripción: {description}")

print("\n" + "="*60)
```

## Comparación de resultados

Ahora comparemos nuestra implementación manual con scikit-learn:

```python
import matplotlib.pyplot as plt

# Ya tenemos nuestra red entrenada (nn) de la sección anterior
# y la red de scikit-learn (mlp)

# Obtener predicciones de ambas
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[0], [1], [1], [0]])

# Nuestra implementación
_, probs_manual = nn.predict(X_train)
probs_manual = probs_manual.flatten()

# Scikit-learn
probs_sklearn = mlp.predict_proba(X_train)[:, 1]

# Comparación visual
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ===== SUBPLOT 1: PREDICCIONES COMPARADAS =====
ax1 = axes[0, 0]

x_pos = np.arange(len(X_train))
width = 0.35
input_labels = [f'{tuple(x)}' for x in X_train]

bars1 = ax1.bar(x_pos - width/2, probs_manual, width,
               label='Nuestra Implementación', alpha=0.7,
               color='blue', edgecolor='black', linewidth=2)
bars2 = ax1.bar(x_pos + width/2, probs_sklearn, width,
               label='Scikit-learn', alpha=0.7,
               color='green', edgecolor='black', linewidth=2)

ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(input_labels, fontsize=10, fontweight='bold')
ax1.set_xlabel('Entrada', fontsize=11, fontweight='bold')
ax1.set_ylabel('Probabilidad Predicha', fontsize=11, fontweight='bold')
ax1.set_title('Comparación de Predicciones', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1.1)

# Añadir valores
for i, (b1, b2) in enumerate(zip(bars1, bars2)):
    ax1.text(b1.get_x() + b1.get_width()/2, b1.get_height() + 0.03,
            f'{probs_manual[i]:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax1.text(b2.get_x() + b2.get_width()/2, b2.get_height() + 0.03,
            f'{probs_sklearn[i]:.3f}', ha='center', fontsize=9, fontweight='bold')

# ===== SUBPLOT 2: DIFERENCIAS =====
ax2 = axes[0, 1]

differences = np.abs(probs_manual - probs_sklearn)
colors_diff = ['green' if d < 0.01 else 'orange' if d < 0.05 else 'red'
               for d in differences]

bars_diff = ax2.bar(x_pos, differences, color=colors_diff, alpha=0.7,
                    edgecolor='black', linewidth=2)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(input_labels, fontsize=10, fontweight='bold')
ax2.set_xlabel('Entrada', fontsize=11, fontweight='bold')
ax2.set_ylabel('Diferencia Absoluta', fontsize=11, fontweight='bold')
ax2.set_title('Diferencias en Probabilidades\n|Manual - Sklearn|',
             fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar, diff in zip(bars_diff, differences):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.001,
            f'{diff:.4f}', ha='center', fontsize=9, fontweight='bold')

# ===== SUBPLOT 3: CURVAS DE PÉRDIDA =====
ax3 = axes[1, 0]

ax3.plot(nn.history, linewidth=2, color='blue', alpha=0.7,
        label='Nuestra Implementación')
ax3.plot(mlp.loss_curve_, linewidth=2, color='green', alpha=0.7,
        label='Scikit-learn')
ax3.axhline(y=0.01, color='red', linestyle='--', linewidth=2, alpha=0.5)

ax3.set_xlabel('Época', fontsize=11, fontweight='bold')
ax3.set_ylabel('Pérdida (MSE)', fontsize=11, fontweight='bold')
ax3.set_title('Curvas de Pérdida Comparadas', fontsize=12, fontweight='bold')
ax3.set_yscale('log')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, which='both')

# ===== SUBPLOT 4: TABLA COMPARATIVA =====
ax4 = axes[1, 1]
ax4.axis('off')

# Crear tabla comparativa
comparison_data = [
    ['Métrica', 'Nuestra Impl.', 'Scikit-learn'],
    ['', '', ''],
    ['Precisión', f'{accuracy:.1f}%', f'{(predictions == y_train.flatten()).mean()*100:.1f}%'],
    ['Épocas', f'{len(nn.history)}', f'{mlp.n_iter_}'],
    ['Pérdida final', f'{nn.history[-1]:.6f}', f'{mlp.loss_:.6f}'],
    ['Líneas código', '~100', '~10'],
    ['Tiempo dev.', 'Alto', 'Bajo'],
    ['Control', 'Total', 'Medio'],
    ['Flexibilidad', 'Alta', 'Media'],
    ['Aprendizaje', '★★★★★', '★★☆☆☆'],
]

# Colores por fila
row_colors = ['lightblue'] + ['white'] + ['lightyellow']*7

table = ax4.table(cellText=comparison_data, cellLoc='center',
                 loc='center', bbox=[0, 0, 1, 1],
                 cellColours=[['lightgray']*3] + [['white']*3] +
                             [[c, c, c] for c in row_colors[2:]])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Estilo especial para encabezados
for i in range(3):
    cell = table[(0, i)]
    cell.set_text_props(weight='bold', fontsize=11)
    cell.set_facecolor('darkblue')
    cell.set_text_props(color='white')

ax4.set_title('Tabla Comparativa Completa', fontsize=12, fontweight='bold', pad=20)

plt.suptitle('Nuestra Implementación vs. Scikit-Learn: Comparación Detallada',
            fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()
```

### Análisis de las diferencias

```python
print("\n" + "="*60)
print("ANÁLISIS DE DIFERENCIAS")
print("="*60)

print(f"\n1. PRECISIÓN:")
print(f"   Nuestra implementación: {accuracy:.1f}%")
print(f"   Scikit-learn:          {(predictions == y_train.flatten()).mean()*100:.1f}%")
print(f"   → Ambas resuelven XOR perfectamente ✓")

print(f"\n2. PROBABILIDADES:")
avg_diff = np.mean(differences)
max_diff = np.max(differences)
print(f"   Diferencia promedio:    {avg_diff:.6f}")
print(f"   Diferencia máxima:      {max_diff:.6f}")
print(f"   → Prácticamente idénticas ✓")

print(f"\n3. CONVERGENCIA:")
print(f"   Nuestra implementación: {len(nn.history)} épocas")
print(f"   Scikit-learn:          {mlp.n_iter_} épocas")
print(f"   → Velocidad similar")

print(f"\n4. CÓDIGO:")
print(f"   Nuestra implementación: ~100 líneas")
print(f"   Scikit-learn:          ~10 líneas")
print(f"   → Scikit-learn 10x más conciso")

print("\n" + "="*60)
print("CONCLUSIÓN")
print("="*60)
print("""
Ambas implementaciones resuelven XOR perfectamente.

NUESTRA IMPLEMENTACIÓN:
  ✓ Entendemos cada detalle del algoritmo
  ✓ Control total sobre el proceso
  ✓ Base sólida para innovar
  ✗ Más código, más tiempo de desarrollo

SCIKIT-LEARN:
  ✓ Código conciso (10 líneas)
  ✓ Optimizado y testeado
  ✓ Ideal para producción rápida
  ✗ "Caja negra" si no entiendes los fundamentos

RECOMENDACIÓN:
  1. Aprende con implementación manual (como hemos hecho)
  2. Usa scikit-learn para proyectos reales
  3. Conocer ambos te hace un mejor ML practitioner
""")
print("="*60)
```

### Cuándo usar cada enfoque

```python
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

scenarios = [
    ("Aprendiendo ML/DL", "Manual ★★★★★", "Sklearn ★★☆☆☆",
     "La implementación manual enseña los fundamentos"),
    
    ("Prototipo rápido", "Manual ★★☆☆☆", "Sklearn ★★★★★",
     "Scikit-learn permite iterar rápidamente"),
    
    ("Proyecto producción", "Manual ★☆☆☆☆", "Sklearn ★★★★★",
     "Usa librerías testeadas y mantenidas"),
    
    ("Investigación nueva", "Manual ★★★★★", "Sklearn ★★☆☆☆",
     "Necesitas modificar el algoritmo"),
    
    ("Debugging problema", "Manual ★★★★★", "Sklearn ★★☆☆☆",
     "Entender qué falla requiere conocer detalles"),
    
    ("Dataset grande", "Manual ★☆☆☆☆", "Sklearn ★★★★☆",
     "Sklearn está optimizado (o usa PyTorch/TF)"),
]

y_pos = 0.95
ax.text(0.5, y_pos, '¿Cuándo usar cada enfoque?', ha='center', fontsize=16,
       fontweight='bold', transform=ax.transAxes)

y_pos = 0.88
header_color = 'lightblue'
ax.text(0.15, y_pos, 'Escenario', ha='center', fontsize=12, fontweight='bold',
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor=header_color))
ax.text(0.45, y_pos, 'Manual', ha='center', fontsize=12, fontweight='bold',
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor=header_color))
ax.text(0.70, y_pos, 'Sklearn', ha='center', fontsize=12, fontweight='bold',
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor=header_color))

y_pos = 0.80
for scenario, manual, sklearn, reason in scenarios:
    # Escenario
    ax.text(0.15, y_pos, scenario, ha='center', fontsize=10,
           transform=ax.transAxes, bbox=dict(boxstyle='round',
           facecolor='lightyellow', alpha=0.7))
    
    # Manual score
    manual_color = 'lightgreen' if '★★★★' in manual else 'lightyellow' if '★★★' in manual else 'lightcoral'
    ax.text(0.45, y_pos, manual, ha='center', fontsize=10,
           transform=ax.transAxes, bbox=dict(boxstyle='round',
           facecolor=manual_color, alpha=0.7))
    
    # Sklearn score
    sklearn_color = 'lightgreen' if '★★★★' in sklearn else 'lightyellow' if '★★★' in sklearn else 'lightcoral'
    ax.text(0.70, y_pos, sklearn, ha='center', fontsize=10,
           transform=ax.transAxes, bbox=dict(boxstyle='round',
           facecolor=sklearn_color, alpha=0.7))
    
    # Razón
    ax.text(0.5, y_pos - 0.04, reason, ha='center', fontsize=8,
           style='italic', transform=ax.transAxes, color='gray')
    
    y_pos -= 0.12

ax.text(0.5, 0.02, 'Conclusión: Conocer ambos enfoques te hace más versátil',
       ha='center', fontsize=11, fontweight='bold', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()
```

---

**Recapitulación:**
- ✅ **Scikit-learn resuelve XOR en ~10 líneas** vs. nuestras ~100 líneas
- ✅ **Resultados prácticamente idénticos**: ambas logran 100% precisión
- ✅ **Diferencias mínimas** en probabilidades predichas
- ✅ **Código conciso** pero pierde transparencia
- 🎯 **El conocimiento es acumulativo**: entender la implementación manual te hace mejor usuario de scikit-learn

**Moraleja:** Aprende con implementaciones manuales, produce con librerías profesionales. ¡El mejor ingeniero conoce ambos mundos!

En la siguiente y última sección, cerraremos con conclusiones y próximos pasos en tu camino de aprendizaje.

# 9. Conclusiones

Hemos completado un viaje extraordinario: desde entender por qué un simple problema lógico desafió a los pioneros de la IA, hasta implementar una solución completa desde cero.

## Lo que hemos aprendido

### El panorama completo

```python
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Título
ax.text(0.5, 0.95, '🎓 Tu Viaje de Aprendizaje', ha='center', fontsize=18,
       fontweight='bold', transform=ax.transAxes)

# Conceptos aprendidos
concepts = [
    ("1. El Problema XOR",
     "Por qué un perceptrón simple falla\nNecesidad de no-linealidad",
     "🧩"),
    
    ("2. Arquitectura MLP",
     "2 entradas → 4 ocultas → 1 salida\nPesos, sesgos y función sigmoide",
     "🏗️"),
    
    ("3. Forward Propagation",
     "Cómo fluyen los datos\nCombinación lineal + activación",
     "→"),
    
    ("4. Función de Coste",
     "Medir el error con MSE\nGuía para el aprendizaje",
     "📊"),
    
    ("5. Backpropagation",
     "Distribuir la 'culpa' del error\nRegla de la Cadena en acción",
     "←"),
    
    ("6. Descenso del Gradiente",
     "Actualizar pesos iterativamente\nLearning rate: tamaño del paso",
     "⬇️"),
    
    ("7. Implementación Completa",
     "~100 líneas de código funcional\nDe teoría a práctica",
     "💻"),
    
    ("8. Scikit-Learn",
     "Misma solución en 10 líneas\nPoder de las librerías",
     "📚"),
]

y_pos = 0.85
for i, (title, description, emoji) in enumerate(concepts):
    # Color alternado
    color = 'lightblue' if i % 2 == 0 else 'lightgreen'
    
    # Emoji
    ax.text(0.08, y_pos, emoji, ha='center', fontsize=24,
           transform=ax.transAxes)
    
    # Título
    ax.text(0.15, y_pos + 0.01, title, ha='left', fontsize=11,
           fontweight='bold', transform=ax.transAxes)
    
    # Descripción
    ax.text(0.15, y_pos - 0.02, description, ha='left', fontsize=9,
           style='italic', transform=ax.transAxes, color='gray')
    
    # Línea de fondo
    rect = plt.Rectangle((0.05, y_pos - 0.035), 0.9, 0.07,
                         facecolor=color, alpha=0.3, transform=ax.transAxes)
    ax.add_patch(rect)
    
    y_pos -= 0.09

plt.tight_layout()
plt.show()
```

### Las habilidades que has desarrollado

```python
print("="*70)
print("✅ HABILIDADES DESARROLLADAS")
print("="*70)

skills = {
    "Conceptuales": [
        "• Entiendes por qué las redes neuronales necesitan capas ocultas",
        "• Comprendes el rol de las funciones de activación no lineales",
        "• Sabes cómo se propaga el error hacia atrás (backpropagation)",
        "• Entiendes la importancia del learning rate",
    ],
    
    "Técnicas": [
        "• Puedes implementar una red neuronal desde cero con NumPy",
        "• Sabes calcular gradientes usando la regla de la cadena",
        "• Puedes entrenar redes con descenso del gradiente",
        "• Dominas el uso de MLPClassifier de scikit-learn",
    ],
    
    "Prácticas": [
        "• Visualizas curvas de pérdida para diagnosticar entrenamiento",
        "• Interpretas probabilidades y fronteras de decisión",
        "• Comparas implementaciones manuales vs. librerías",
        "• Debuggeas problemas de convergencia",
    ],
    
    "Fundamentales": [
        "• Ya no ves las redes neuronales como 'magia negra'",
        "• Tienes intuición sobre qué ocurre durante el entrenamiento",
        "• Puedes explicar backpropagation sin usar ecuaciones complejas",
        "• Entiendes el trade-off entre control y conveniencia",
    ]
}

for category, skill_list in skills.items():
    print(f"\n{category}:")
    print("-" * 70)
    for skill in skill_list:
        print(f"  {skill}")

print("\n" + "="*70)
```

### El logro principal

```python
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# Caja principal
ax.text(0.5, 0.7, '🎯 EL LOGRO PRINCIPAL', ha='center', fontsize=16,
       fontweight='bold', transform=ax.transAxes)

achievement_text = """
No has memorizado fórmulas.
Has desarrollado INTUICIÓN PROFUNDA.

Ahora entiendes:
  → Por qué las capas ocultas transforman el espacio
  → Cómo el gradiente guía el aprendizaje
  → Por qué la sigmoide añade no-linealidad
  → Cuándo usar implementación manual vs. librería

Este conocimiento es tu SUPERPODER:
mientras otros usan frameworks como cajas negras,
tú entiendes qué ocurre dentro.
"""

ax.text(0.5, 0.35, achievement_text, ha='center', va='center', fontsize=11,
       transform=ax.transAxes, family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightyellow',
                edgecolor='gold', linewidth=3, alpha=0.9))

plt.tight_layout()
plt.show()
```

## Próximos pasos

### Tu camino de aprendizaje continúa

```python
print("\n" + "="*70)
print("🚀 PRÓXIMOS PASOS EN TU CAMINO")
print("="*70)

learning_path = {
    "📘 Nivel 1: Consolidar (1-2 semanas)": [
        "→ Implementa XOR con arquitectura 2-8-1 (más neuronas)",
        "→ Prueba con learning rates diferentes (0.1, 1.0, 2.0)",
        "→ Añade visualización de pesos durante el entrenamiento",
        "→ Experimenta con inicialización de pesos (Xavier, He)",
    ],
    
    "📗 Nivel 2: Expandir (1 mes)": [
        "→ Resuelve otros problemas: AND+OR combinados, círculos concéntricos",
        "→ Implementa otras funciones de activación: tanh, ReLU",
        "→ Añade momentum al descenso del gradiente",
        "→ Prueba con datasets reales: Iris, Digits (MNIST simple)",
    ],
    
    "📙 Nivel 3: Profundizar (2-3 meses)": [
        "→ Implementa redes más profundas (3-4 capas)",
        "→ Añade regularización L2 para evitar overfitting",
        "→ Implementa mini-batch gradient descent",
        "→ Estudia optimizadores avanzados: Adam, RMSprop",
    ],
    
    "📕 Nivel 4: Especializarse (3-6 meses)": [
        "→ Explora CNNs para visión por computadora",
        "→ Aprende RNNs/LSTMs para series temporales",
        "→ Domina PyTorch o TensorFlow para proyectos grandes",
        "→ Participa en competencias Kaggle",
    ],
}

for level, steps in learning_path.items():
    print(f"\n{level}")
    print("-" * 70)
    for step in steps:
        print(f"  {step}")

print("\n" + "="*70)
```

### Recursos recomendados

```python
print("\n" + "="*70)
print("📚 RECURSOS PARA CONTINUAR")
print("="*70)

resources = {
    "Libros": [
        "• 'Neural Networks and Deep Learning' - Michael Nielsen",
        "  → Online gratis, excelente para fundamentos",
        "• 'Deep Learning' - Goodfellow, Bengio, Courville",
        "  → El libro definitivo, más matemático",
        "• 'Hands-On Machine Learning' - Aurélien Géron",
        "  → Muy práctico, con mucho código",
    ],
    
    "Cursos Online": [
        "• Fast.ai - Practical Deep Learning",
        "  → Top-down, muy hands-on",
        "• Deep Learning Specialization - Andrew Ng (Coursera)",
        "  → Fundamentos sólidos, bien explicado",
        "• CS231n Stanford - Convolutional Neural Networks",
        "  → Para visión por computadora",
    ],
    
    "Práctica": [
        "• Kaggle - Datasets y competencias reales",
        "• Papers With Code - Implementaciones recientes",
        "• Tu propio GitHub - Construye un portfolio",
    ],
    
    "Comunidad": [
        "• r/MachineLearning, r/learnmachinelearning",
        "• Discord/Slack de ML (muchas comunidades activas)",
        "• Twitter/X - Sigue investigadores (@karpathy, etc.)",
    ],
}

for category, items in resources.items():
    print(f"\n{category}:")
    print("-" * 70)
    for item in items:
        print(f"  {item}")

print("\n" + "="*70)
```

### Consejos finales

```python
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

ax.text(0.5, 0.95, '💡 CONSEJOS PARA TU CAMINO', ha='center', fontsize=16,
       fontweight='bold', transform=ax.transAxes)

tips = [
    ("🏗️ CONSTRUYE", "No te limites a leer o ver videos.\nImplementa, experimenta, rompe cosas."),
    ("🐛 DEBUGGEA", "Aprenderás más depurando errores\nque leyendo teoría perfecta."),
    ("📊 VISUALIZA", "Siempre visualiza datos, predicciones,\ny curvas de pérdida."),
    ("📝 DOCUMENTA", "Escribe sobre lo que aprendes.\nEnseñar consolida el conocimiento."),
    ("🎯 PROYECTOS", "Trabaja en problemas que te apasionen,\nno solo tutoriales."),
    ("🔄 ITERA", "Versión 1 imperfecta hoy >\nversión perfecta que nunca terminas."),
    ("⏰ CONSTANCIA", "30 minutos diarios > 5 horas\nun domingo cada dos meses."),
    ("🤝 COMPARTE", "Contribuye a open source,\nayuda a otros, crece en comunidad."),
]

y_pos = 0.85
for emoji, title, description in tips:
    # Emoji grande
    ax.text(0.08, y_pos, emoji, ha='center', fontsize=20,
           transform=ax.transAxes)
    
    # Título en negrita
    ax.text(0.15, y_pos + 0.01, title, ha='left', fontsize=12,
           fontweight='bold', transform=ax.transAxes)
    
    # Descripción
    ax.text(0.15, y_pos - 0.025, description, ha='left', fontsize=9,
           style='italic', transform=ax.transAxes, color='dimgray')
    
    y_pos -= 0.10

plt.tight_layout()
plt.show()
```

### El mensaje final

```python
print("\n" + "="*70)
print("🎉 FELICIDADES")
print("="*70)
print("""
Has completado este tutorial sobre redes neuronales multicapa.
Pero esto es solo EL COMIENZO de tu viaje en Deep Learning.

LO QUE HAS LOGRADO:
  ✓ Entiendes los fundamentos de cómo aprenden las redes neuronales
  ✓ Puedes implementar backpropagation desde cero
  ✓ Resolviste un problema históricamente importante (XOR)
  ✓ Tienes intuición sobre forward/backward propagation
  ✓ Sabes cuándo usar implementación manual vs. librerías

LO QUE VIENE DESPUÉS:
  → Redes más profundas (Deep Learning)
  → Arquitecturas especializadas (CNNs, RNNs, Transformers)
  → Problemas del mundo real (visión, lenguaje, etc.)
  → Tu propio proyecto de impacto

RECUERDA:
  "El experto en algo fue una vez un principiante."
  
Cada desarrollador de IA empezó exactamente donde estás ahora.
La diferencia está en dar el SIGUIENTE paso.

¿Cuál será el tuyo?
""")
print("="*70)
```

### Visualización final: tu progreso

```python
fig, ax = plt.subplots(figsize=(12, 6))

# Eje de progreso
ax.plot([0, 10], [0, 0], 'k-', linewidth=3)

# Puntos en el camino
milestones = [
    (0, "Inicio\n(No sabías nada\nde redes neuronales)", 'red'),
    (5, "Ahora\n(Entiendes MLP\ny backpropagation)", 'green'),
    (10, "Futuro\n(Experto en\nDeep Learning)", 'blue'),
]

for x, label, color in milestones:
    ax.plot(x, 0, 'o', markersize=20, color=color,
           markeredgecolor='black', markeredgewidth=2, zorder=3)
    ax.text(x, -0.5, label, ha='center', fontsize=11, fontweight='bold')

# Marcador "Estás aquí"
ax.annotate('ESTÁS AQUÍ →', xy=(5, 0), xytext=(5, 1.5),
           arrowprops=dict(arrowstyle='->', color='green', lw=3),
           fontsize=14, fontweight='bold', color='green', ha='center')

# Flecha de progreso
ax.annotate('', xy=(9.5, 0.3), xytext=(5.5, 0.3),
           arrowprops=dict(arrowstyle='->', color='blue', lw=4))
ax.text(7.5, 0.8, 'Tu próximo paso', ha='center', fontsize=12,
       fontweight='bold', color='blue')

ax.set_xlim(-1, 11)
ax.set_ylim(-1, 2)
ax.axis('off')
ax.set_title('Tu Viaje en Deep Learning', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()
```

---

## Palabras finales

```python
print("\n" + "="*70)
print(" " * 20 + "¡GRACIAS POR APRENDER CON ESTE TUTORIAL!")
print("="*70)
print("""
Si este artículo te ha sido útil:

  ⭐ Guárdalo como referencia
  💬 Comparte tus aprendizajes en los comentarios
  🔄 Recomiéndalo a otros que estén aprendiendo ML
  🚀 ¡Construye algo increíble con este conocimiento!

El código completo está disponible en este notebook.
Siéntete libre de experimentar, modificar y mejorar.

Nos vemos en el siguiente nivel del Deep Learning.

¡Mucho éxito en tu camino! 🎓🚀
""")
print("="*70)
print("\n")
```

---

**Has completado el tutorial. Ahora el futuro del AI está en tus manos. ¡Adelante! 🚀**